# Kenya Gazette PDF to structured JSON (Docling + Spatial Reorder)

This notebook runs **Docling** on one or more Gazette PDFs and builds a **single JSON-friendly record per file** with:

- PDF metadata: inferred title, file name, size in bytes, page count
- Full **Docling** exports: plain text, Markdown, and a compact document summary (full `export_to_dict` is optional — it can be large)
- **Spatial reading-order correction**: reorders text elements using bounding-box coordinates to fix two-column reading order (left column top-to-bottom, then right column, then full-width zones)
- **Gazette notices** split by the phrase `GAZETTE NOTICE NO.` (Kenya-style), with notice number, header line, and full notice text

**Input:** drop PDFs into the `pdfs/` folder next to this notebook (or change `PDF_DIR` below). Use **Choose which PDFs** to run on every file or only a chosen subset.

> **See also:** `gazette_docling_pipeline.ipynb` for the original pipeline without spatial reordering.

## Environment

Use the project virtual environment (already created in this folder as `.venv`):

- **Windows (PowerShell):** `.\.venv\Scripts\Activate.ps1`
- **Then:** `python -m pip install -r requirements.txt` if needed
- **Jupyter kernel:** register with `python -m ipykernel install --user --name=gazette-docling --display-name="Python (gazette-docling)"` using the venv’s `python`.

Select that kernel in this notebook before running cells.

In [1]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from docling.document_converter import DocumentConverter
from docling_core.types.doc.labels import DocItemLabel

# Project root = folder containing this notebook
PROJECT_ROOT = Path.cwd().resolve()
PDF_DIR = PROJECT_ROOT / "pdfs"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PDF_DIR:", PDF_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PDF_DIR: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\pdfs
OUTPUT_DIR: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output


In [2]:
# ---------------------------------------------------------------------------
# F11: Masthead Parser — Extract identity fields from Gazette masthead
# ---------------------------------------------------------------------------

def parse_masthead(text: str) -> dict:
    """Parse Kenya Gazette masthead to extract volume, issue_no, publication_date, supplement_no.
    
    Args:
        text: First ~30 lines of plain_spatial text from the PDF.
        
    Returns:
        dict with keys: volume (str|None), issue_no (int|None), 
        publication_date (str "YYYY-MM-DD"|None), supplement_no (int|None)
        
    Rules:
        - Unparseable field -> None. Never invent values. Never raise.
        - Volume: Roman numeral verbatim (e.g., "CXXIV")
        - Issue: Integer parsed from "No. X" pattern
        - Date: Normalized to ISO format "YYYY-MM-DD" (strip ordinals like "23rd")
        - Supplement: Integer if "Supplement No. X" or "-SX" detected
    """
    import re
    from typing import Optional
    
    result = {
        "volume": None,
        "issue_no": None,
        "publication_date": None,
        "supplement_no": None,
    }
    
    # Get first 30 lines for scanning
    lines = text.split("\n")[:30]
    text_block = "\n".join(lines)
    
    # Pattern 1: Vol. CXXIV-No. 282 (most common)
    # Pattern 2: Vol. CXXIV No. 282
    # Pattern 3: Vol. CXXIV-No. 282 NAIROBI, 23rd December, 2022
    vol_issue_pattern = re.compile(
        r"Vol\.?\s*([IVXLC]+)\s*-?\s*No\.?\s*(\d+)",
        re.IGNORECASE
    )
    
    # Supplement pattern: "Supplement No. X" or "-SX" suffix
    supplement_pattern = re.compile(
        r"(?:Supplement\s+No\.?\s*(\d+)|-S(\d+))",
        re.IGNORECASE
    )
    
    # Date pattern: NAIROBI, 23rd December, 2022 or NAIROBI, 23 December 2022
    # Also handle variations: "30th December, 2005", "3rd June, 2025"
    date_pattern = re.compile(
        r"(?:NAIROBI|Nairobi)\s*,?\s*(\d{1,2})(?:st|nd|rd|th)?\s+([A-Za-z]+),?\s*(\d{4})",
        re.IGNORECASE
    )
    
    # Extract volume and issue number
    vol_match = vol_issue_pattern.search(text_block)
    if vol_match:
        result["volume"] = vol_match.group(1).upper()  # Preserve Roman numeral
        result["issue_no"] = int(vol_match.group(2))
    
    # Extract supplement if present
    supp_match = supplement_pattern.search(text_block)
    if supp_match:
        supp_num = supp_match.group(1) or supp_match.group(2)
        if supp_num:
            result["supplement_no"] = int(supp_num)
    
    # Extract and normalize date
    date_match = date_pattern.search(text_block)
    if date_match:
        day = date_match.group(1)
        month_str = date_match.group(2).lower()
        year = date_match.group(3)
        
        # Month name to number mapping
        months = {
            "january": "01", "february": "02", "march": "03",
            "april": "04", "may": "05", "june": "06",
            "july": "07", "august": "08", "september": "09",
            "october": "10", "november": "11", "december": "12",
            "jan": "01", "feb": "02", "mar": "03", "apr": "04",
            "jun": "06", "jul": "07", "aug": "08", "sep": "09",
            "sept": "09", "oct": "10", "nov": "11", "dec": "12"
        }
        
        month = months.get(month_str)
        if month:
            # Pad day with leading zero if needed
            day_padded = day.zfill(2)
            result["publication_date"] = f"{year}-{month}-{day_padded}"
    
    return result


# Test helper for verification
def test_parse_masthead():
    """Run test cases from F11 spec."""
    test_dir = Path("output")
    
    test_cases = [
        ("Kenya Gazette Vol CXXIVNo 282", {"volume": "CXXIV", "issue_no": 282, "publication_date": "2022-12-23", "supplement_no": None}),
        ("Kenya Gazette Vol CVIINo 90 - pre 2010", {"volume": "CVII", "issue_no": 90, "publication_date": "2005-12-30", "supplement_no": None}),
        ("Kenya Gazette Vol CXXVIINo 111", {"volume": "CXXVII", "issue_no": 111, "publication_date": "2025-06-03", "supplement_no": None}),
        ("Kenya Gazette Vol CXXVIINo 63", {"volume": "CXXVII", "issue_no": 63, "publication_date": "2025-03-28", "supplement_no": None}),
    ]
    
    print("F11 Test Results:")
    print("-" * 60)
    
    for folder_name, expected in test_cases:
        # Find the spatial.txt file
        folder = test_dir / folder_name
        txt_file = folder / f"{folder_name}_spatial.txt"
        
        if not txt_file.exists():
            # Try alternate naming
            txt_files = list(folder.glob("*_spatial.txt"))
            if txt_files:
                txt_file = txt_files[0]
        
        if txt_file.exists():
            with open(txt_file, "r", encoding="utf-8") as f:
                text = f.read()
            
            result = parse_masthead(text)
            
            # Check each field
            status = "PASS"
            for key in ["volume", "issue_no", "publication_date", "supplement_no"]:
                if result.get(key) != expected[key]:
                    status = "FAIL"
                    break
            
            print(f"{folder_name[:40]:40} | {status}")
            if status == "FAIL":
                print(f"  Expected: {expected}")
                print(f"  Got:      {result}")
        else:
            print(f"{folder_name[:40]:40} | SKIP (no file)")
    
    print("-" * 60)


# Uncomment to run tests:
# test_parse_masthead()


## Gazette notice splitting

Notices are detected by a **strict full-line, all-caps** match on `GAZETTE NOTICE NO. <digits>` (with tolerance for the OCR typo `GAZETE`). This avoids false positives from inline references like "IN Gazette Notice No. 14152 …" that appear inside prose.

Each notice is then structured with:
- **`title_lines`** — heading lines before the statutory body (e.g. act name, chapter reference, subtitle)
- **`body_segments`** — typed blocks (`text`, `table`, `blank`) detected via multi-space column heuristics
- **`derived_table`** — optional structured rows when an S/No–Name–Position table is detected
- **Running-header stripping** — page furniture (page numbers, running headers, dates) is removed from notice bodies

**Corrigenda** (correction notices) are extracted separately from the preamble section before the first notice. Each corrigendum captures the referenced notice number/year and the error/correction text.

You can tune `NOTICE_HEAD_RE` if your issues use a different wording.

In [3]:
# ---------------------------------------------------------------------------
# Strict notice-header regex: full-line, all-caps only.
# Handles the common OCR typo "GAZETE" (single T).
# This avoids false positives from inline references like
# "IN Gazette Notice No. 14152 …" which are mixed-case / mid-sentence.
# ---------------------------------------------------------------------------
NOTICE_HEAD_RE = re.compile(
    r"^(?:GAZETTE|GAZETE)\s+NOTICE\.?\s+NO\.?\s*(\d+)\s*$"
)

# Recovered-header regex: matches a notice header embedded in a noisy line
# (pipe-separated financial tables, leading prose, or trailing content).
# Intentionally all-caps to avoid mid-sentence "Gazette Notice No. 14152"
# corrigenda references. See docs/known-issues.md items 8 and 14.
RECOVERED_HEAD_RE = re.compile(
    r"(?:^|\s|\|)\s*(?:GAZETTE|GAZETE)\s+NOTICE\.?\s+NO\.?\s*(\d+)(?:\s|\|)"
)

# Lines that are running headers / page furniture to strip from notice bodies.
# Matched anywhere in the body (not just edges) -- see docs/known-issues.md item 6.
RUNNING_HEADER_RES = [
    re.compile(r"^\s*THE KENYA GAZETTE\s*$", re.I),
    re.compile(r"^\s*\d{1,2}(?:st|nd|rd|th)\s+\w+,?\s+\d{4}\s*$"),
    re.compile(r"^\s*\d{4,5}\s*$"),
    re.compile(r"^\s*CONTENTS\s*$", re.I),
    re.compile(r"^\s*Published\s+by\s+Authority\s+of\s+the\s+Republic\s+of\s+Kenya\s*$", re.I),
    re.compile(r"^\s*Price\s+Sh\.?\s*\d+[\.,]?\d*\s*$", re.I),
    re.compile(r"^\s*\(Registered\s+as\s+a\s+Newspaper\s+at\s+the\s+G\.?P\.?O\.?\)\s*$", re.I),
    re.compile(r"^\s*Vol\.?\s*[IVXLC]+\s*-+\s*No\.?\s*\d+\s*$", re.I),
    re.compile(r"^\s*NAIROBI\s*,?\s+\d{1,2}(?:st|nd|rd|th)?\s+\w+,?\s+\d{4}\s*$", re.I),
]

# Patterns that mark the start of the statutory body (ending the title region).
BODY_START_RE = re.compile(
    r"^(PURSUANT|NOTICE\s+is|NOTICE\s+OF|IT\s+IS\s+|IN\s+EXERCISE\s+|"
    r"IN\s+PURSUANCE\s+|WHEREAS\s+|TAKE\s+NOTICE\s+|THE\s+following\s+)",
    re.I,
)


def _strip_running_headers(lines: list[str]) -> list[str]:
    return [ln for ln in lines if not any(r.match(ln) for r in RUNNING_HEADER_RES)]


def _split_on_multiple_spaces(line: str) -> list[str]:
    return [p for p in re.split(r" {2,}", line.strip()) if p]


def _extract_title_stack(body_lines: list[str]) -> tuple[list[str], list[str]]:
    """Heading lines run from the start of the notice body until a line that clearly
    begins statutory text (PURSUANT, IN EXERCISE, IT IS, …)."""
    titles: list[str] = []
    for idx, ln in enumerate(body_lines):
        s = ln.strip()
        if not s:
            if titles:
                return titles, body_lines[idx + 1:]
            continue
        if BODY_START_RE.match(s):
            return titles, body_lines[idx:]
        titles.append(ln)
    return titles, []


def _segment_body_lines(lines: list[str]) -> list[dict[str, Any]]:
    """Split notice body into text vs table-like blocks.
    Consecutive lines that split into 2+ columns (separated by 2+ spaces) become a table."""
    blocks: list[dict[str, Any]] = []
    i = 0
    n = len(lines)

    while i < n:
        ln = lines[i]
        if not ln.strip():
            blocks.append({"type": "blank", "lines": []})
            i += 1
            continue

        parts = _split_on_multiple_spaces(ln)
        is_table_start = (
            (len(parts) >= 2 and not re.match(r"^[A-Z][a-z]", (ln.strip()[:20] or "")))
            or re.match(r"^S\.?\s*/?\s*No\.?", ln.strip(), re.I)
        )
        if is_table_start:
            table_lines: list[str] = []
            j = i
            while j < n:
                row = lines[j]
                if not row.strip():
                    break
                p2 = _split_on_multiple_spaces(row)
                if len(p2) < 2 and j > i:
                    if len(table_lines) >= 3 and len(row) > 120:
                        break
                if len(p2) >= 2:
                    table_lines.append(row)
                    j += 1
                    continue
                if table_lines and len(row) < 100:
                    table_lines.append(row)
                    j += 1
                    continue
                break

            if len(table_lines) >= 2:
                rows = [_split_on_multiple_spaces(x) for x in table_lines]
                blocks.append({"type": "table", "raw_lines": table_lines, "rows": rows})
                i = j
                continue

        para: list[str] = [ln]
        i += 1
        while i < n:
            nxt = lines[i]
            if not nxt.strip():
                break
            p2 = _split_on_multiple_spaces(nxt)
            if len(p2) >= 2:
                break
            para.append(nxt)
            i += 1
        blocks.append({"type": "text", "lines": para})

    return blocks


_TWO_SERIAL_RE = re.compile(r"^\s*(\d+)\s+(\d+)\.?\s*$")


def _repair_merged_rows(rows: list[dict[str, str]]) -> tuple[list[dict[str, str]], list[str]]:
    """Split rows where adjacent serial numbers were fused (e.g. "625 626").

    See docs/known-issues.md section 11. Returns (repaired_rows, reasons).
    """
    repaired: list[dict[str, str]] = []
    reasons: list[str] = []
    for row in rows:
        s = (row.get("s_no") or "").strip()
        m = _TWO_SERIAL_RE.match(s)
        if not m:
            repaired.append(row)
            continue
        a, b = int(m.group(1)), int(m.group(2))
        if b - a != 1:
            repaired.append(row)
            continue
        name = (row.get("name") or "").strip()
        pos = (row.get("position") or "").strip()
        split_name = re.split(r"(?<=\S)\s{2,}(?=\S)", name, maxsplit=1)
        split_pos = re.split(r"(?<=\S)\s{2,}(?=\S)", pos, maxsplit=1)
        name_a = split_name[0] if split_name else name
        name_b = split_name[1] if len(split_name) > 1 else ""
        pos_a = split_pos[0] if split_pos else pos
        pos_b = split_pos[1] if len(split_pos) > 1 else ""
        repaired.append({"s_no": str(a), "name": name_a, "position": pos_a})
        repaired.append({"s_no": str(b), "name": name_b, "position": pos_b})
        reasons.append(f"split merged row {a}/{b}")
    return repaired, reasons


def _try_parse_s_no_table(lines: list[str]) -> dict[str, Any] | None:
    """Recover rows when PDF text has split table columns into separate lines, e.g.:
        S/No. Name
        Position
        1
        Jane Doe
        Chairperson
    """
    start = None
    for i, ln in enumerate(lines):
        s = ln.strip()
        if re.match(r"^S/No\.?\s+Name", s, re.I):
            start = i
            break
    if start is None:
        return None

    i = start + 1
    if i >= len(lines) or lines[i].strip().lower() != "position":
        return None

    rows: list[dict[str, str]] = []
    i += 1
    while i < len(lines):
        block = lines[i].strip()
        if not block or not (re.match(r"^\d+\.?$", block) or _TWO_SERIAL_RE.match(block)):
            break
        idx = block.rstrip(".")
        i += 1
        if i >= len(lines):
            break
        name = lines[i].strip()
        i += 1
        if i >= len(lines):
            rows.append({"s_no": idx, "name": name, "position": ""})
            break
        position = lines[i].strip()
        if position.lower().startswith("as ") and len(position) > 80:
            rows.append({"s_no": idx, "name": name, "position": ""})
            break
        i += 1
        rows.append({"s_no": idx, "name": name, "position": position})

    if not rows:
        return None

    repaired, repair_reasons = _repair_merged_rows(rows)
    result: dict[str, Any] = {
        "format": "s_no_name_position_lines",
        "rows": repaired,
    }
    if repair_reasons:
        result["repairs"] = repair_reasons
    return result


_TERMINAL_PUNCT = (".", "!", "?", '"', "'", ")", "]")


def _ends_with_terminal_punct(text: str) -> bool:
    s = text.rstrip()
    if not s:
        return False
    return s[-1] in _TERMINAL_PUNCT


def _find_recovered_boundaries(
    raw_lines: list[str],
    strict_boundaries: list[tuple[int, str]],
    min_gap_lines: int = 40,
) -> list[tuple[int, str, str]]:
    """Scan gaps between strict boundaries for recovered header matches.

    Returns boundaries as (line_idx, notice_no, header_kind) where header_kind is
    either "strict" or "recovered". Recovered headers are only emitted inside
    large gaps where the strict regex found nothing (likely a financial-table or
    pipe-separated page, known-issues items 8 and 14).
    """
    merged: list[tuple[int, str, str]] = [(i, n, "strict") for i, n in strict_boundaries]
    if not merged:
        return merged

    gaps: list[tuple[int, int]] = []
    for bi in range(len(merged) - 1):
        a = merged[bi][0]
        b = merged[bi + 1][0]
        if b - a > min_gap_lines:
            gaps.append((a + 1, b))
    last = merged[-1][0]
    if len(raw_lines) - last > min_gap_lines:
        gaps.append((last + 1, len(raw_lines)))

    recovered: list[tuple[int, str, str]] = []
    seen_nums: set[str] = {n for _, n, _ in merged}
    for lo, hi in gaps:
        for i in range(lo, hi):
            line = raw_lines[i]
            for m in RECOVERED_HEAD_RE.finditer(line):
                num = m.group(1)
                if num in seen_nums:
                    continue
                recovered.append((i, num, "recovered"))
                seen_nums.add(num)
                break

    merged.extend(recovered)
    merged.sort(key=lambda t: t[0])
    return merged


def _stitch_multipage_notices(notices: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Merge notices that end mid-sentence into the next block when that block
    has no strict header of its own. See docs/known-issues.md item 3.

    Only stitches when the *next* entry has a recovered or missing header --
    strict headers are always kept as boundaries.
    """
    if not notices:
        return notices
    out: list[dict[str, Any]] = []
    for entry in notices:
        if not out:
            out.append(entry)
            continue
        prev = out[-1]
        prev_prov = prev.get("provenance") or {}
        curr_prov = entry.get("provenance") or {}
        prev_text = prev.get("gazette_notice_full_text") or ""
        prev_ends_clean = _ends_with_terminal_punct(prev_text)
        curr_is_recovered = curr_prov.get("header_match") == "recovered"
        if not prev_ends_clean and curr_is_recovered:
            stitched_from = list(prev_prov.get("stitched_from") or [])
            stitched_from.append(entry.get("gazette_notice_no"))
            prev["gazette_notice_full_text"] = (
                prev_text + "\n" + (entry.get("gazette_notice_full_text") or "")
            ).strip()
            prev.setdefault("body_segments", []).extend(entry.get("body_segments") or [])
            prev_prov["stitched_from"] = stitched_from
            prev["provenance"] = prev_prov
            prev_other = prev.setdefault("other_attributes", {})
            entry_other = entry.get("other_attributes") or {}
            prev_other["char_span_end_line"] = entry_other.get(
                "char_span_end_line", prev_other.get("char_span_end_line")
            )
            prev_other["lines_in_body"] = prev_other.get("lines_in_body", 0) + (
                entry_other.get("lines_in_body") or 0
            )
            continue
        out.append(entry)
    return out


def detect_trailing_content_cutoff(text: str, last_notice_start_line: int) -> int | None:
    """Detect where actual notice content ends and trailing material begins.
    
    Trailing content includes subscription pricing, classified ads, index pages,
    and other non-notice material that appears after the last valid gazette notice.
    
    Args:
        text: Full plain text of the document
        last_notice_start_line: Line index where the last notice header appears
        
    Returns:
        Line index where trailing content begins, or None if no trailing content detected
    """
    if not text or not text.strip():
        return None
    
    lines = text.splitlines()
    
    patterns = [
        (r'^\s*NOW\s+ON\s+SALE\b', 'publications_catalog', 10),
        (r'SUBSCRIPTION\s+(?:AND\s+)?ADVERTISEMENT\s+CHARGES', 'subscription', 20),
        (r'SUBSCRIPTION\s+CHARGES', 'subscription', 20),
        (r'IMPORTANT\s+NOTICE\s+TO\s+SUBSCRIBERS', 'important_notice', 20),
        (r'Government\s+Printer\.?\s*$', 'govt_printer', 20),
        (r'^(?:INDEX|CONTENTS)\s*$', 'index', 20),
        (r'CLASSIFIED\s+(?:ADVERTISEMENT|ADS)', 'classified', 20),
    ]
    
    earliest_cutoff = None
    
    for i in range(len(lines)):
        if i <= last_notice_start_line:
            continue
            
        line_stripped = lines[i].strip()
        
        for pattern, label, min_distance in patterns:
            if i - last_notice_start_line < min_distance:
                continue
                
            if re.search(pattern, line_stripped, re.I):
                if earliest_cutoff is None or i < earliest_cutoff:
                    earliest_cutoff = i
                break
    
    return earliest_cutoff


def split_gazette_notices(full_text: str) -> list[dict[str, Any]]:
    """Split flat text into structured notice blocks.

    Uses strict full-line, all-caps header matching first to avoid false
    positives from inline references. A second pass then scans gaps for
    recovered headers embedded in noisy lines (pipe-separated financial tables
    or lines with prefixes). Each notice gets title extraction, body
    segmentation, running-header stripping, optional derived-table recovery,
    and a provenance block recording header_match kind and line span.
    """
    if not full_text or not full_text.strip():
        return []

    raw_lines = full_text.splitlines()

    strict: list[tuple[int, str]] = []
    for i, line in enumerate(raw_lines):
        m = NOTICE_HEAD_RE.match(line.strip())
        if m:
            strict.append((i, m.group(1)))

    boundaries = _find_recovered_boundaries(raw_lines, strict)

    if not boundaries:
        return [
            {
                "gazette_notice_no": None,
                "gazette_notice_header": None,
                "title_lines": [],
                "gazette_notice_full_text": full_text.strip(),
                "body_segments": [],
                "other_attributes": {
                    "reason": "no GAZETTE NOTICE NO. markers found; entire text returned as one block",
                },
                "provenance": {"header_match": "none", "line_span": [0, len(raw_lines)]},
            }
        ]

    notices: list[dict[str, Any]] = []
    for bi, (start_idx, num, kind) in enumerate(boundaries):
        end_idx = boundaries[bi + 1][0] if bi + 1 < len(boundaries) else len(raw_lines)
        body_start = start_idx + 1 if kind == "strict" else start_idx
        chunk = _strip_running_headers(raw_lines[body_start:end_idx])

        titles, body_only = _extract_title_stack(chunk)
        body_for_segments = body_only if body_only else chunk
        segments = _segment_body_lines(body_for_segments)
        derived = _try_parse_s_no_table(chunk)

        header_line = raw_lines[start_idx].strip()
        body_text = "\n".join(chunk).strip()

        if kind == "strict":
            display_header = header_line
            notice_text = f"{header_line}\n{body_text}" if body_text else header_line
        else:
            display_header = f"GAZETTE NOTICE NO. {num}"
            notice_text = f"{display_header}\n{body_text}" if body_text else display_header

        entry: dict[str, Any] = {
            "gazette_notice_no": num,
            "gazette_notice_header": display_header,
            "title_lines": [t.strip() for t in titles],
            "gazette_notice_full_text": notice_text,
            "body_segments": segments,
            "other_attributes": {
                "char_span_start_line": start_idx,
                "char_span_end_line": end_idx,
                "lines_in_body": len(chunk),
            },
            "provenance": {
                "header_match": kind,
                "line_span": [start_idx, end_idx],
                "raw_header_line": header_line,
                "stitched_from": [],
            },
        }
        if derived:
            entry["derived_table"] = derived
        notices.append(entry)

    notices = _stitch_multipage_notices(notices)
    
    if notices and full_text:
        last_notice = notices[-1]
        last_prov = last_notice.get("provenance") or {}
        last_line_span = last_prov.get("line_span") or [0, 0]
        last_notice_start = last_line_span[0]
        last_notice_original_end = last_line_span[1]
        
        cutoff = detect_trailing_content_cutoff(full_text, last_notice_start)
        
        if cutoff is not None and cutoff < last_notice_original_end:
            raw_lines = full_text.splitlines()
            
            last_kind = last_prov.get("header_match", "strict")
            body_start_line = last_notice_start + 1 if last_kind == "strict" else last_notice_start
            
            chunk = _strip_running_headers(raw_lines[body_start_line:cutoff])
            header_line = raw_lines[last_notice_start].strip()
            body_text = "\n".join(chunk).strip()
            
            if last_kind == "strict":
                display_header = header_line
                notice_text = f"{header_line}\n{body_text}" if body_text else header_line
            else:
                num = last_notice.get("gazette_notice_no")
                display_header = f"GAZETTE NOTICE NO. {num}"
                notice_text = f"{display_header}\n{body_text}" if body_text else display_header
            
            last_notice["gazette_notice_full_text"] = notice_text
            last_notice["body_segments"] = _segment_body_lines(chunk)
            
            last_other = last_notice.setdefault("other_attributes", {})
            last_other["char_span_end_line"] = cutoff
            last_other["lines_in_body"] = len(chunk)
            
            last_prov["line_span"] = [last_notice_start, cutoff]
            last_notice["provenance"] = last_prov
    
    return notices


# ---------------------------------------------------------------------------
# Corrigenda extraction: captures correction notices from the CORRIGENDA section
# that appear before the numbered gazette notices.
# ---------------------------------------------------------------------------
CORRIGENDUM_RE = re.compile(
    r"(?:IN\s+)?Gazette\s+Notice\s+No\.?\s*(\d+)\s+of\s+(\d{4})",
    re.I,
)

# Character class matching any of: ASCII double quote, ASCII apostrophe,
# and the four Unicode smart quotes (left/right double, left/right single).
_QUOTE_CLASS = "[\"'\u201c\u201d\u2018\u2019]"

AMEND_PATTERN_RE = re.compile(
    r"amend\s+(?:the\s+)?(.+?)\s+(?:printed\s+as\s+)?"
    + _QUOTE_CLASS + r"(.+?)" + _QUOTE_CLASS
    + r"\s+to\s+read\s+"
    + _QUOTE_CLASS + r"(.+?)" + _QUOTE_CLASS,
    re.I | re.DOTALL,
)


def extract_corrigenda(full_text: str) -> list[dict[str, Any]]:
    """Extract corrigenda (correction notices) from the preamble section.
    
    Corrigenda appear before the first GAZETTE NOTICE NO. header and reference
    other notice numbers with corrections like:
    'IN Gazette Notice No. 14152 of 2025, amend the expression printed as "X" to read "Y"'
    """
    if not full_text or not full_text.strip():
        return []

    raw_lines = full_text.splitlines()

    # Find first notice header to delimit the preamble/corrigenda section
    first_notice_idx = len(raw_lines)
    for i, line in enumerate(raw_lines):
        if NOTICE_HEAD_RE.match(line.strip()):
            first_notice_idx = i
            break

    preamble_text = "\n".join(raw_lines[:first_notice_idx])

    # Find CORRIGENDA section if it exists
    corrigenda_start = None
    for i, line in enumerate(raw_lines[:first_notice_idx]):
        if re.match(r"^\s*CORRIGEND[AE]\s*$", line.strip(), re.I):
            corrigenda_start = i
            break

    if corrigenda_start is None:
        # No explicit CORRIGENDA header; scan preamble for corrigendum patterns
        corrigenda_text = preamble_text
    else:
        corrigenda_text = "\n".join(raw_lines[corrigenda_start:first_notice_idx])

    # Split on "IN Gazette Notice No." or similar patterns to find individual corrigenda
    corrigenda: list[dict[str, Any]] = []

    # Find all corrigendum statements
    segments = re.split(r"(?=(?:IN\s+)?Gazette\s+Notice\s+No\.?\s*\d)", corrigenda_text, flags=re.I)

    for seg in segments:
        seg = seg.strip()
        if not seg:
            continue

        # Extract referenced notice number and year
        ref_match = CORRIGENDUM_RE.search(seg)
        if not ref_match:
            continue

        referenced_no = ref_match.group(1)
        referenced_year = ref_match.group(2)

        # Extract correction details
        amend_match = AMEND_PATTERN_RE.search(seg)
        if amend_match:
            what_field = amend_match.group(1).strip()
            error_text = amend_match.group(2).strip()
            correction_text = amend_match.group(3).strip()
        else:
            what_field = None
            error_text = None
            correction_text = None

        # Clean up raw text (join multi-line, normalize whitespace)
        raw_text = " ".join(seg.split())

        corrigenda.append({
            "referenced_notice_no": referenced_no,
            "referenced_year": referenced_year,
            "what_corrected": what_field,
            "error_text": error_text,
            "correction_text": correction_text,
            "raw_text": raw_text,
        })

    return corrigenda


def extract_title_from_docling(doc) -> str:
    for item in getattr(doc, "texts", []) or []:
        if getattr(item, "label", None) == DocItemLabel.TITLE and getattr(item, "text", None):
            return str(item.text).strip()
    return ""


def docling_export_summary(doc_dict: dict[str, Any]) -> dict[str, Any]:
    """Small fingerprint of the Docling JSON without dumping huge trees twice."""
    texts = doc_dict.get("texts") or []
    return {
        "schema_name": doc_dict.get("schema_name"),
        "version": doc_dict.get("version"),
        "name": doc_dict.get("name"),
        "texts_count": len(texts) if isinstance(texts, list) else None,
        "tables_count": len(doc_dict.get("tables") or []) if isinstance(doc_dict.get("tables"), list) else None,
        "pictures_count": len(doc_dict.get("pictures") or []) if isinstance(doc_dict.get("pictures"), list) else None,
        "pages_count": len(doc_dict.get("pages") or []) if isinstance(doc_dict.get("pages"), list) else None,
    }


_GAZETTE_NOTICE_MD_LINE = re.compile(
    r"^(\#\# )?(GAZETTE NOTICE NO\. \d+)\s*$",
    re.MULTILINE,
)

_GAZETTE_NOTICE_HIGHLIGHT_STYLE = (
    'style="background-color:#fff3cd;color:#1a1a1a;padding:0.15em 0.35em;border-radius:3px;font-weight:600;"'
)


def highlight_gazette_notices_in_markdown(md: str) -> str:
    """Wrap standalone GAZETTE NOTICE NO. lines for visible highlight in Markdown HTML preview."""

    def repl(m: re.Match) -> str:
        notice = m.group(2)
        inner = f'<span {_GAZETTE_NOTICE_HIGHLIGHT_STYLE}>{notice}</span>'
        if m.group(1):
            return f"## {inner}"
        return inner

    return _GAZETTE_NOTICE_MD_LINE.sub(repl, md)

## Confidence scoring

Rule-based confidence for every extracted notice along five dimensions:

- **`notice_number`** -- is the captured number a plausible digit string?
- **`structure`** -- does the body look complete (length, legal markers, signature)?
- **`spatial`** -- does the text flow read cleanly (no mid-sentence breaks, no repeated chunks)?
- **`boundary`** -- was the header a strict match, does the span end with punctuation?
- **`table`** -- for notices that produced a `derived_table`, are the rows well-formed?

Every scorer returns `(score, reasons)`. The reasons list is what makes low scores auditable.

The `composite` is a weighted mean (see `docs/data-quality-confidence-scoring.md`). `table` is only folded in when a `derived_table` is present.

Two thresholds drive downstream use:

- `composite >= 0.80` -- high confidence, usable without review.
- `0.50 <= composite < 0.80` -- medium, spot-check.
- `composite < 0.50` -- likely broken, needs review (and is the gate for optional LLM semantic validation).

In [4]:
# ---------------------------------------------------------------------------
# Confidence scorers.
# Every scorer returns (score, reasons). Reasons are short strings explaining
# deductions so a low composite is auditable. Scorers are stateless and
# import-ready for a later gazette_pipeline package.
# ---------------------------------------------------------------------------

_LEGAL_MARKERS = (
    "IN EXERCISE", "IT IS NOTIFIED", "IN PURSUANCE", "PURSUANT TO",
    "WHEREAS", "TAKE NOTICE", "Dated the", "(Cap. ", "(No. ",
)
_SIGNATURE_RE = re.compile(r"^\s*[A-Z][A-Z .,'-]{4,}\s*,?\s*$", re.M)
_DATE_LINE_RE = re.compile(
    r"\bDated\s+the\s+\d{1,2}(?:st|nd|rd|th)?\s+\w+,?\s+\d{4}", re.I
)


def _clip(v: float) -> float:
    return max(0.0, min(1.0, float(v)))


def score_notice_number(num: str | None) -> tuple[float, list[str]]:
    reasons: list[str] = []
    if num is None or not str(num).strip():
        return 0.0, ["empty notice number"]
    s = str(num).strip()
    if re.fullmatch(r"\d{2,6}", s):
        return 1.0, []
    if re.fullmatch(r"\d+/\d{4}", s):
        reasons.append("notice number in year/number form")
        return 0.6, reasons
    if re.fullmatch(r"\d+[A-Za-z]", s):
        reasons.append("notice number has trailing letter")
        return 0.5, reasons
    if re.fullmatch(r"\d{7,}", s):
        reasons.append("notice number is unusually long")
        return 0.4, reasons
    if re.fullmatch(r"\d{1}", s):
        reasons.append("single-digit notice number is suspicious")
        return 0.3, reasons
    if re.fullmatch(r"[^\d]+", s):
        reasons.append("notice number has no digits")
        return 0.1, reasons
    reasons.append("notice number has unexpected shape")
    return 0.3, reasons


def score_structure(
    title_lines: list[str],
    body_segments: list[dict[str, Any]],
    derived_table: dict[str, Any] | None,
    text: str,
) -> tuple[float, list[str]]:
    reasons: list[str] = []
    score = 1.0
    body_len = len(text or "")
    non_blank_lines = [ln for ln in (text or "").splitlines() if ln.strip()]
    n_lines = len(non_blank_lines)

    if body_len < 50:
        reasons.append(f"body very short ({body_len} chars)")
        score -= 0.6
    elif body_len < 120:
        reasons.append(f"body short ({body_len} chars)")
        score -= 0.2
    elif body_len > 100_000:
        reasons.append(f"body very long ({body_len} chars) -- possible merge")
        score -= 0.3

    if n_lines <= 2:
        reasons.append("only header/no body content")
        score -= 0.4

    has_marker = any(m.lower() in (text or "").lower() for m in _LEGAL_MARKERS)
    has_date = bool(_DATE_LINE_RE.search(text or ""))
    has_sig = bool(_SIGNATURE_RE.search(text or ""))
    has_table = bool(derived_table) or any(
        s.get("type") == "table" for s in (body_segments or [])
    )
    signals = sum([has_marker, has_date, has_sig])

    if signals == 0 and not has_table:
        reasons.append("no legal marker, date, or signature found")
        score -= 0.3
    elif signals <= 1 and not has_table:
        reasons.append("only one structural marker present")
        score -= 0.1

    if not title_lines:
        reasons.append("no title lines extracted")
        score -= 0.05

    return _clip(score), reasons


def score_spatial(text: str) -> tuple[float, list[str]]:
    reasons: list[str] = []
    score = 1.0
    if not text:
        return 0.0, ["empty text"]

    lines = [ln for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2:
        return _clip(score), reasons

    mid_break = 0
    for i in range(len(lines) - 1):
        cur = lines[i].rstrip()
        nxt = lines[i + 1].lstrip()
        if not cur or not nxt:
            continue
        if cur[-1] in _TERMINAL_PUNCT or cur[-1] in ",;:-":
            continue
        if nxt[:1].islower():
            mid_break += 1
    if mid_break >= 3:
        reasons.append(f"{mid_break} mid-sentence breaks (possible column interleaving)")
        score -= min(0.5, 0.1 * mid_break)

    words = re.findall(r"\w+", text.lower())
    n_words = len(words)
    if n_words > 50:
        grams = [" ".join(words[i:i + 6]) for i in range(n_words - 5)]
        from collections import Counter
        counts = Counter(grams)
        repeats = sum(1 for _, c in counts.items() if c >= 2)
        if repeats >= 3:
            reasons.append(f"{repeats} repeated 6-grams (possible column merge)")
            score -= min(0.3, 0.05 * repeats)

    if n_words > 100:
        mid_cap = 0
        for m in re.finditer(r"(?<=[a-z]{3})\s+([A-Z][a-z]{3,})", text):
            mid_cap += 1
        density = mid_cap / max(1, n_words)
        if density > 0.10 and mid_cap > 20:
            reasons.append(
                f"high mid-sentence cap density {density:.2f} "
                f"({mid_cap}/{n_words} words)"
            )
            score -= min(0.15, density * 0.5)

    return _clip(score), reasons


def score_boundary(
    header_match: str,
    text: str,
    line_span: tuple[int, int] | list[int],
    next_line_span: tuple[int, int] | list[int] | None,
) -> tuple[float, list[str]]:
    reasons: list[str] = []
    score = 1.0

    if header_match == "strict":
        pass
    elif header_match == "recovered":
        reasons.append("header recovered from noisy line (capped at 0.6)")
        score = min(score, 0.6)
    elif header_match == "inferred":
        reasons.append("header inferred (capped at 0.4)")
        score = min(score, 0.4)
    else:
        reasons.append("no notice header found")
        score = min(score, 0.2)

    span_len = 0
    if line_span and len(line_span) == 2:
        span_len = int(line_span[1]) - int(line_span[0])
    if span_len <= 1:
        reasons.append("span is a single line")
        score -= 0.4
    elif span_len < 3:
        reasons.append("span is very short")
        score -= 0.2

    ends_clean = _ends_with_terminal_punct(text or "")
    if not ends_clean:
        reasons.append("text does not end with terminal punctuation")
        score -= 0.15

    if next_line_span and len(next_line_span) == 2 and line_span and len(line_span) == 2:
        gap = int(next_line_span[0]) - int(line_span[1])
        if gap > 20:
            reasons.append(f"gap of {gap} lines to next notice -- possible missing content")
            score -= 0.2

    return _clip(score), reasons


def score_table(derived_table: dict[str, Any] | None) -> tuple[float, list[str]]:
    if not derived_table:
        return 1.0, []
    reasons: list[str] = []
    score = 1.0
    rows = derived_table.get("rows") or []
    if not rows:
        return 0.2, ["derived_table has no rows"]

    repairs = derived_table.get("repairs") or []
    if repairs:
        reasons.append(f"{len(repairs)} merged-row repairs applied")
        score -= min(0.3, 0.05 * len(repairs))

    serials: list[int] = []
    for r in rows:
        m = re.match(r"^(\d+)$", str(r.get("s_no") or "").strip())
        if m:
            serials.append(int(m.group(1)))
    if len(serials) >= 2:
        jumps = sum(1 for a, b in zip(serials, serials[1:]) if b - a != 1)
        if jumps:
            reasons.append(f"{jumps} non-sequential serial jumps")
            score -= min(0.3, 0.05 * jumps)

    name_lens = [len((r.get("name") or "").strip()) for r in rows if r.get("name")]
    if name_lens:
        mean = sum(name_lens) / len(name_lens)
        over = sum(1 for n in name_lens if n > mean * 3)
        if over >= max(1, len(rows) // 10):
            reasons.append("some cells dramatically longer than others (likely merged)")
            score -= 0.15

    return _clip(score), reasons


def composite_confidence(scores: dict[str, float]) -> float:
    """Weighted mean of the rule-based dimensions.

    Weights match docs/data-quality-confidence-scoring.md:
    notice_number 0.30, structure 0.25, spatial 0.25, boundary 0.20.
    `table` is folded in (weight 0.15) only when a derived_table was present
    (table score == 1.0 for notices without a table is neutral; we still
    include it at a smaller weight to reward table-heavy notices that parse
    cleanly).
    """
    base = (
        scores.get("notice_number", 0.0) * 0.30
        + scores.get("structure", 0.0) * 0.25
        + scores.get("spatial", 0.0) * 0.25
        + scores.get("boundary", 0.0) * 0.20
    )
    t = scores.get("table")
    if t is None:
        return _clip(base)
    return _clip(base * 0.85 + t * 0.15)


def score_notice(
    entry: dict[str, Any],
    next_entry: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Attach confidence_scores and confidence_reasons to a notice in place."""
    text = entry.get("gazette_notice_full_text") or ""
    prov = entry.get("provenance") or {}
    line_span = prov.get("line_span") or [
        (entry.get("other_attributes") or {}).get("char_span_start_line", 0),
        (entry.get("other_attributes") or {}).get("char_span_end_line", 0),
    ]
    next_line_span = None
    if next_entry is not None:
        np_prov = next_entry.get("provenance") or {}
        next_line_span = np_prov.get("line_span")

    nn_s, nn_r = score_notice_number(entry.get("gazette_notice_no"))
    st_s, st_r = score_structure(
        entry.get("title_lines") or [],
        entry.get("body_segments") or [],
        entry.get("derived_table"),
        text,
    )
    sp_s, sp_r = score_spatial(text)
    bd_s, bd_r = score_boundary(
        prov.get("header_match", "strict"),
        text,
        line_span,
        next_line_span,
    )
    tb_s, tb_r = score_table(entry.get("derived_table"))

    scores: dict[str, float] = {
        "notice_number": round(nn_s, 3),
        "structure": round(st_s, 3),
        "spatial": round(sp_s, 3),
        "boundary": round(bd_s, 3),
    }
    if entry.get("derived_table"):
        scores["table"] = round(tb_s, 3)
    scores["composite"] = round(composite_confidence(scores), 3)

    reasons: list[str] = []
    for prefix, rs in (
        ("notice_number", nn_r), ("structure", st_r), ("spatial", sp_r),
        ("boundary", bd_r), ("table", tb_r if entry.get("derived_table") else []),
    ):
        for msg in rs:
            reasons.append(f"{prefix}: {msg}")

    entry["confidence_scores"] = scores
    entry["confidence_reasons"] = reasons
    return entry


def score_notices(notices: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Score a list of notices; uses next-entry context for boundary scoring."""
    for i, entry in enumerate(notices):
        nxt = notices[i + 1] if i + 1 < len(notices) else None
        score_notice(entry, nxt)
    return notices


def compute_document_confidence(
    notices: list[dict[str, Any]],
    layout_confidence: float = 1.0,
    ocr_quality: float = 1.0,
) -> dict[str, Any]:
    """Aggregate per-notice scores into a document-level summary."""
    counts = {"high": 0, "medium": 0, "low": 0}
    composites: list[float] = []
    for n in notices:
        c = (n.get("confidence_scores") or {}).get("composite")
        if c is None:
            continue
        composites.append(float(c))
        if c >= 0.80:
            counts["high"] += 1
        elif c >= 0.50:
            counts["medium"] += 1
        else:
            counts["low"] += 1
    total = len(composites) or 1
    notice_split = 1.0 - (counts["low"] / total)
    composite = (
        (sum(composites) / len(composites)) * 0.6
        + layout_confidence * 0.2
        + ocr_quality * 0.2
    ) if composites else (layout_confidence * 0.5 + ocr_quality * 0.5)
    return {
        "layout": round(_clip(layout_confidence), 3),
        "ocr_quality": round(_clip(ocr_quality), 3),
        "notice_split": round(_clip(notice_split), 3),
        "composite": round(_clip(composite), 3),
        "counts": counts,
        "mean_composite": round(sum(composites) / len(composites), 3) if composites else 0.0,
        "min_composite": round(min(composites), 3) if composites else 0.0,
        "n_notices": len(notices),
    }


print("Confidence scorers ready.")

Confidence scorers ready.


In [5]:
# ---------------------------------------------------------------------------
# Downstream helpers -- let any analysis inherit the confidence of its inputs.
# ---------------------------------------------------------------------------


def filter_notices(
    notices: list[dict[str, Any]],
    min_composite: float = 0.70,
) -> list[dict[str, Any]]:
    """Return only notices whose composite meets the threshold."""
    out: list[dict[str, Any]] = []
    for n in notices:
        c = (n.get("confidence_scores") or {}).get("composite")
        if c is not None and float(c) >= min_composite:
            out.append(n)
    return out


def partition_by_band(
    notices: list[dict[str, Any]],
) -> dict[str, list[dict[str, Any]]]:
    """Group notices by confidence band (high / medium / low / unscored)."""
    bands: dict[str, list[dict[str, Any]]] = {
        "high": [], "medium": [], "low": [], "unscored": []
    }
    for n in notices:
        c = (n.get("confidence_scores") or {}).get("composite")
        if c is None:
            bands["unscored"].append(n)
        elif c >= 0.80:
            bands["high"].append(n)
        elif c >= 0.50:
            bands["medium"].append(n)
        else:
            bands["low"].append(n)
    return bands


def aggregate_confidence(notices: list[dict[str, Any]]) -> dict[str, float]:
    """Summarize a notice list so analyses can quote a single confidence number.

    Returns mean, min, and the fraction in the high band. Use this whenever
    you report a statistic computed from notices -- the caller can multiply
    their claim by `fraction_high` or note the mean alongside the result.
    """
    cs: list[float] = [
        float((n.get("confidence_scores") or {}).get("composite"))
        for n in notices
        if (n.get("confidence_scores") or {}).get("composite") is not None
    ]
    if not cs:
        return {"mean": 0.0, "min": 0.0, "fraction_high": 0.0, "n": 0}
    return {
        "mean": round(sum(cs) / len(cs), 3),
        "min": round(min(cs), 3),
        "fraction_high": round(sum(1 for c in cs if c >= 0.80) / len(cs), 3),
        "n": len(cs),
    }


def explain(notice: dict[str, Any]) -> str:
    """Human-readable triage view for a single notice."""
    header = notice.get("gazette_notice_header") or "(no header)"
    no = notice.get("gazette_notice_no") or "?"
    scores = notice.get("confidence_scores") or {}
    reasons = notice.get("confidence_reasons") or []
    prov = notice.get("provenance") or {}
    llm = notice.get("llm_validation") or {}
    parts = [
        f"Notice {no}: {header}",
        f"  header_match={prov.get('header_match', '?')} "
        f"line_span={prov.get('line_span')} "
        f"stitched_from={prov.get('stitched_from') or []}",
        "  scores: " + ", ".join(f"{k}={v}" for k, v in scores.items()),
    ]
    if reasons:
        parts.append("  reasons:")
        for r in reasons:
            parts.append(f"    - {r}")
    if llm:
        parts.append(
            f"  llm: needs_review={llm.get('needs_human_review')} "
            f"issues={llm.get('issues') or []}"
        )
    return "\n".join(parts)


print("Downstream helpers ready: filter_notices, partition_by_band, aggregate_confidence, explain.")

Downstream helpers ready: filter_notices, partition_by_band, aggregate_confidence, explain.


## Spatial reading-order correction

Docling's internal reading-order predictor uses heuristics that can misorder text in two-column layouts — a notice that starts at the bottom of the left column and continues at the top of the right column may be split or interleaved with unrelated content.

The `reorder_by_spatial_position()` function fixes this by using each element's bounding-box coordinates from the Docling export dict:

1. **Group** all text (and table) elements by page number
2. **Detect columns** by checking whether element centers fall left or right of the page midpoint
3. **Find the column-zone boundary** — the y-coordinate where the shorter column ends
4. **Reclassify** elements below that boundary as full-width (even if left-aligned)
5. **Sort**: furniture (headers) → left column top-to-bottom → right column top-to-bottom → full-width zone top-to-bottom
6. **Concatenate** to produce corrected plain text

In [6]:
from dataclasses import dataclass as _dc


@_dc
class _BBoxElement:
    """Lightweight carrier for a text/table element with its spatial position."""
    page_no: int
    label: str
    content_layer: str
    text: str
    left: float
    top: float
    right: float
    bottom: float
    center_x: float
    self_ref: str

    @property
    def width(self) -> float:
        return self.right - self.left


def _table_to_text(data: dict[str, Any]) -> str:
    """Build a pipe-delimited text representation of a table from its grid data."""
    grid = data.get("grid")
    if not grid:
        cells = data.get("table_cells") or []
        return " | ".join(c.get("text", "") for c in cells if c.get("text"))
    rows: list[str] = []
    for row in grid:
        cols = [cell.get("text", "") for cell in row]
        rows.append(" | ".join(cols))
    return "\n".join(rows)


def _extract_elements(doc_dict: dict[str, Any]) -> list[_BBoxElement]:
    """Pull every text and table element that has provenance bbox data."""
    elements: list[_BBoxElement] = []

    for item in doc_dict.get("texts") or []:
        provs = item.get("prov") or []
        if not provs:
            continue
        prov = provs[0]
        bbox = prov.get("bbox")
        if not bbox:
            continue
        text = item.get("text") or item.get("orig") or ""
        if not text.strip():
            continue
        l, t, r, b = bbox["l"], bbox["t"], bbox["r"], bbox["b"]
        elements.append(_BBoxElement(
            page_no=prov["page_no"],
            label=item.get("label", "text"),
            content_layer=item.get("content_layer", "body"),
            text=text,
            left=l, top=t, right=r, bottom=b,
            center_x=(l + r) / 2,
            self_ref=item.get("self_ref", ""),
        ))

    for item in doc_dict.get("tables") or []:
        provs = item.get("prov") or []
        if not provs:
            continue
        prov = provs[0]
        bbox = prov.get("bbox")
        if not bbox:
            continue
        text = _table_to_text(item.get("data") or {})
        if not text.strip():
            continue
        l, t, r, b = bbox["l"], bbox["t"], bbox["r"], bbox["b"]
        elements.append(_BBoxElement(
            page_no=prov["page_no"],
            label="table",
            content_layer=item.get("content_layer", "body"),
            text=text,
            left=l, top=t, right=r, bottom=b,
            center_x=(l + r) / 2,
            self_ref=item.get("self_ref", ""),
        ))

    return elements


def _get_page_dimensions(doc_dict: dict[str, Any]) -> dict[int, tuple[float, float]]:
    """Return {page_no: (width, height)} from the pages map."""
    dims: dict[int, tuple[float, float]] = {}
    pages = doc_dict.get("pages") or {}
    for _key, pinfo in pages.items():
        sz = pinfo.get("size") or {}
        pno = pinfo.get("page_no")
        if pno is not None and sz:
            dims[pno] = (sz.get("width", 595.0), sz.get("height", 842.0))
    return dims


# Minimum fraction of page text-area width for an element to count as full-width
_FULLWIDTH_RATIO = 0.55
# How far above the first centered/full-width anchor to extend the zone boundary
_FW_TRANSITION_TOLERANCE = 50.0


def _reorder_page(
    page_elements: list[_BBoxElement],
    page_width: float,
) -> list[_BBoxElement]:
    """Reorder elements on a single page: left col -> right col -> full-width bottom."""

    mid_x = page_width / 2.0
    text_area_width = page_width - 100

    furniture: list[_BBoxElement] = []
    left_candidates: list[_BBoxElement] = []
    right_candidates: list[_BBoxElement] = []
    full_width: list[_BBoxElement] = []

    for el in page_elements:
        if el.content_layer == "furniture":
            furniture.append(el)
            continue

        clearly_right = el.left >= mid_x
        clearly_left = el.right <= mid_x
        spans_both = el.left < mid_x and el.right > mid_x
        wide_enough = el.width > text_area_width * _FULLWIDTH_RATIO
        centered = (spans_both
                    and not wide_enough
                    and abs(el.center_x - mid_x) < 80
                    and el.width < text_area_width * 0.45)

        if spans_both and wide_enough:
            full_width.append(el)
        elif centered:
            full_width.append(el)
        elif clearly_right:
            right_candidates.append(el)
        elif clearly_left:
            left_candidates.append(el)
        elif el.center_x < mid_x:
            left_candidates.append(el)
        else:
            right_candidates.append(el)

    # --- Single-column shortcut ---
    # If only one side of the page has candidates the page is not
    # two-column.  Merge everything and sort top-to-bottom.
    if not left_candidates or not right_candidates:
        all_body = left_candidates + right_candidates + full_width
        all_body.sort(key=lambda el: -el.top)
        return furniture + all_body

    # --- Detect the full-width zone transition ---
    # Use centered / spanning elements as anchors: the topmost such body
    # element marks where the full-width zone begins.  Everything at or
    # below that level (with tolerance) is reclassified out of the columns.
    fw_body = [el for el in full_width if el.content_layer != "furniture"]
    fw_transition_y = None

    if fw_body:
        fw_transition_y = max(el.top for el in fw_body)

        # Guard: only apply if the anchor is in the lower portion of the
        # page (below the median of column elements).  If centered content
        # sits above most column text it is likely a page-level heading,
        # not a bottom full-width zone.
        col_tops = [el.top for el in left_candidates + right_candidates]
        if col_tops:
            col_tops_sorted = sorted(col_tops, reverse=True)
            median_top = col_tops_sorted[len(col_tops_sorted) // 2]
            if fw_transition_y >= median_top:
                fw_transition_y = None

    if fw_transition_y is not None:
        threshold = fw_transition_y + _FW_TRANSITION_TOLERANCE

        left_col: list[_BBoxElement] = []
        for el in left_candidates:
            if el.top <= threshold:
                full_width.append(el)
            else:
                left_col.append(el)

        right_col: list[_BBoxElement] = []
        for el in right_candidates:
            if el.top <= threshold:
                full_width.append(el)
            else:
                right_col.append(el)
    else:
        # No full-width anchors (or anchors are at page top): fall back to
        # the "shorter column" heuristic.
        left_col = list(left_candidates)
        right_col = list(right_candidates)

        if left_col and right_col:
            left_bottom = min(el.bottom for el in left_col)
            right_bottom = min(el.bottom for el in right_col)
            col_zone_bottom = max(left_bottom, right_bottom)

            left_col = [el for el in left_col if el.top >= col_zone_bottom]
            full_width.extend(
                el for el in left_candidates if el.top < col_zone_bottom
            )
            right_col = [el for el in right_col if el.top >= col_zone_bottom]
            full_width.extend(
                el for el in right_candidates if el.top < col_zone_bottom
            )

    # Sort each group top-to-bottom (BOTTOMLEFT origin: higher t = higher on page)
    key_y = lambda el: -el.top
    furniture.sort(key=key_y)
    left_col.sort(key=key_y)
    right_col.sort(key=key_y)
    full_width.sort(key=key_y)

    return furniture + left_col + right_col + full_width


def _cluster_y_bands(
    page_elements: list[_BBoxElement],
    gap_multiplier: float = 1.5,
) -> list[list[_BBoxElement]]:
    """Cluster elements into y-bands separated by vertical gaps.

    Bands correspond to horizontal strips that can have their own layout
    (1-col, 2-col, or full-width) independently of the rest of the page.
    PDF y origin is bottom-left: higher `top` = higher on the page, so we
    walk elements in descending `top` order.
    """
    body = [el for el in page_elements if el.content_layer != "furniture"]
    if not body:
        return []
    sorted_els = sorted(body, key=lambda e: -e.top)
    bands: list[list[_BBoxElement]] = []
    current: list[_BBoxElement] = [sorted_els[0]]
    for el in sorted_els[1:]:
        prev_band_bottom = min(b.bottom for b in current)
        height = max((b.top - b.bottom) for b in current) or 10.0
        gap = prev_band_bottom - el.top
        if gap > height * gap_multiplier:
            bands.append(current)
            current = [el]
        else:
            current.append(el)
    bands.append(current)
    return bands


def _classify_band(
    band: list[_BBoxElement],
    page_width: float,
) -> tuple[str, float]:
    """Classify a band as 'one_col', 'two_col', 'full_width', or 'mixed'.

    Returns (label, band_confidence). Confidence is high when the band is
    unambiguous (clear 2-col split, or all elements spanning full width).
    """
    if not band:
        return "empty", 1.0
    mid = page_width / 2.0
    text_width = page_width - 100
    wide = [el for el in band if el.width > text_width * _FULLWIDTH_RATIO]
    if len(wide) >= max(1, len(band) // 2):
        if len(wide) == len(band):
            return "full_width", 1.0
        return "full_width", 0.8

    clearly_left = sum(1 for el in band if el.right <= mid)
    clearly_right = sum(1 for el in band if el.left >= mid)
    ambiguous = len(band) - clearly_left - clearly_right - len(wide)

    if clearly_left == 0 or clearly_right == 0:
        total_clear = clearly_left + clearly_right
        conf = 1.0 - (ambiguous / max(1, len(band)))
        return "one_col", max(0.5, conf)

    total = len(band)
    clear = clearly_left + clearly_right
    conf = clear / total
    if ambiguous / total > 0.3:
        return "mixed", max(0.3, conf)
    return "two_col", conf


def compute_page_layout_confidence(
    page_elements: list[_BBoxElement],
    page_width: float,
) -> dict[str, Any]:
    """Per-page layout confidence summary.

    Addresses docs/known-issues.md items 1, 2, 10, 13: mixed layouts and
    narrow centered headers produce ambiguous classifications. A band-level
    view surfaces this as a confidence number instead of hiding it inside a
    best-effort global split.
    """
    bands = _cluster_y_bands(page_elements)
    if not bands:
        return {"layout_confidence": 1.0, "bands": [], "mode": "empty"}
    band_infos: list[dict[str, Any]] = []
    weighted_conf = 0.0
    total_weight = 0
    modes: dict[str, int] = {}
    for band in bands:
        label, conf = _classify_band(band, page_width)
        weight = len(band)
        band_infos.append({"mode": label, "confidence": round(conf, 3), "n_elements": weight})
        weighted_conf += conf * weight
        total_weight += weight
        modes[label] = modes.get(label, 0) + 1
    avg = weighted_conf / max(1, total_weight)
    mode_label = max(modes.items(), key=lambda kv: kv[1])[0] if modes else "unknown"
    if len(modes) > 1:
        mode_label = f"hybrid ({mode_label} dominant)"
    return {
        "layout_confidence": round(avg, 3),
        "bands": band_infos,
        "mode": mode_label,
        "n_bands": len(bands),
    }


def reorder_by_spatial_position_with_confidence(
    doc_dict: dict[str, Any],
) -> tuple[str, dict[str, Any]]:
    """Reorder text elements by spatial position AND emit a layout-confidence
    report.

    Returns (plain_text, info) where info contains per-page layout confidence
    and a document-level aggregate.
    """
    elements = _extract_elements(doc_dict)
    dims = _get_page_dimensions(doc_dict)

    by_page: dict[int, list[_BBoxElement]] = {}
    for el in elements:
        by_page.setdefault(el.page_no, []).append(el)

    page_texts: list[str] = []
    page_infos: list[dict[str, Any]] = []
    weighted_sum = 0.0
    weight_total = 0
    for page_no in sorted(by_page):
        pw, _ph = dims.get(page_no, (595.0, 842.0))
        page_els = by_page[page_no]
        info = compute_page_layout_confidence(page_els, pw)
        info["page_no"] = page_no
        page_infos.append(info)
        ordered = _reorder_page(page_els, pw)
        page_texts.append(
            "\n\n".join(el.text for el in ordered if el.content_layer != "furniture")
        )
        n_body = sum(1 for el in page_els if el.content_layer != "furniture")
        weighted_sum += info["layout_confidence"] * n_body
        weight_total += n_body

    doc_layout_conf = weighted_sum / max(1, weight_total) if weight_total else 1.0
    return (
        "\n\n".join(page_texts),
        {
            "layout_confidence": round(doc_layout_conf, 3),
            "n_pages": len(page_infos),
            "pages": page_infos,
        },
    )


def reorder_by_spatial_position(doc_dict: dict[str, Any]) -> str:
    """Backward-compatible wrapper: reorder and return only plain text.

    For two-column pages the reading order becomes:
      page header -> left column (top-to-bottom) -> right column (top-to-bottom)
      -> full-width zone at page bottom (top-to-bottom)
    Single-column pages are simply sorted top-to-bottom.
    """
    text, _info = reorder_by_spatial_position_with_confidence(doc_dict)
    return text


print("Spatial reorder function ready (with band-level layout confidence).")

Spatial reorder function ready (with band-level layout confidence).


## Convert pipeline (with spatial reorder)

`DocumentConverter` is created once and reused. For each PDF we populate the output record and optionally write the full Docling JSON to `output/<stem>_docling.json`.

Gazette notices are now split from the **spatially reordered** plain text instead of the raw `export_to_text()` output. Both versions are kept in the record for comparison.

In [7]:
# F13: Identity field helpers
# F14: Envelope versioning helpers

import hashlib
from datetime import datetime, timezone

# F14: Module-level version constants
LIBRARY_VERSION = "0.1.0"
SCHEMA_VERSION = "1.0"


def make_extracted_at() -> str:
    """Return current UTC timestamp as ISO 8601 string with Z suffix.
    
    Format: YYYY-MM-DDTHH:MM:SSZ (whole-second precision).
    Never raises: returns Unix epoch string on any failure.
    """
    try:
        return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception:
        return "1970-01-01T00:00:00Z"


def compute_pdf_sha256(pdf_path: Path) -> str:
    """Compute SHA-256 hex digest of PDF file bytes.
    
    Returns lowercase hex string (length 64). On read failure, returns
    fallback string for stability.
    """
    try:
        pdf_bytes = pdf_path.read_bytes()
        return hashlib.sha256(pdf_bytes).hexdigest()
    except Exception:
        return f"unknown_{pdf_path.name}"


def make_gazette_issue_id(masthead: dict, pdf_sha256: str) -> tuple[str, bool]:
    """Build canonical gazette issue ID from masthead fields.
    
    Returns (issue_id, is_fallback). Canonical form:
        KE-GAZ-{volume}-{issue_no}-{publication_date}[-S{n}]
    Fallback (when required fields missing):
        KE-GAZ-UNKNOWN-{pdf_sha256[:12]}
    
    Required fields: volume, issue_no, publication_date
    Optional field: supplement_no (appended as -S{n} when present and non-zero)
    """
    volume = masthead.get("volume")
    issue_no = masthead.get("issue_no")
    pub_date = masthead.get("publication_date")
    supplement_no = masthead.get("supplement_no")
    
    # Check if required fields are present
    if volume is None or issue_no is None or pub_date is None:
        fallback_id = f"KE-GAZ-UNKNOWN-{pdf_sha256[:12]}"
        return (fallback_id, True)
    
    # Build canonical ID
    issue_id = f"KE-GAZ-{volume}-{issue_no}-{pub_date}"
    
    # Append supplement suffix if present and non-zero
    if supplement_no is not None and supplement_no != 0:
        issue_id = f"{issue_id}-S{supplement_no}"
    
    return (issue_id, False)


def make_notice_id(gazette_issue_id: str, gazette_notice_no: Any, line_span_start: int) -> str:
    """Build notice ID from gazette issue ID and notice number.
    
    For keyed notices (gazette_notice_no is not None):
        {gazette_issue_id}:{gazette_notice_no}
    For orphan blocks (gazette_notice_no is None):
        {gazette_issue_id}:_orphan_{line_span_start}
    
    line_span_start defaults to 0 if None.
    """
    if line_span_start is None:
        line_span_start = 0
    
    if gazette_notice_no is None:
        return f"{gazette_issue_id}:_orphan_{line_span_start}"
    else:
        return f"{gazette_issue_id}:{gazette_notice_no}"


In [8]:
import hashlib

from kenya_gazette_parser.models import Envelope


def _estimate_ocr_quality(doc_dict: dict[str, Any], plain_text: str) -> tuple[float, list[str]]:
    """Heuristic OCR-quality score for the whole document.

    Low score = likely a scanned pre-2010 gazette with garbled text. See
    docs/known-issues.md item 7. Signals combined:
      - Fraction of non-letter gibberish characters in extracted text.
      - Fraction of very short (<3 char) text elements in Docling output.
      - Ratio of extractable text length to expected (pages x ~2000 chars).
    """
    reasons: list[str] = []
    if not plain_text:
        return 0.0, ["no extractable text"]

    letters = sum(1 for c in plain_text if c.isalpha())
    total = sum(1 for c in plain_text if not c.isspace())
    letter_ratio = letters / max(1, total)
    if letter_ratio < 0.65:
        reasons.append(f"low letter ratio {letter_ratio:.2f} (possible OCR noise)")

    texts = doc_dict.get("texts") or []
    if texts:
        short_elements = sum(
            1 for t in texts if len((t.get("text") or "").strip()) <= 2
        )
        short_ratio = short_elements / len(texts)
        if short_ratio > 0.25:
            reasons.append(
                f"{short_ratio:.0%} of text elements are 1-2 chars (OCR fragments)"
            )
    else:
        short_ratio = 0.0

    pages = doc_dict.get("pages") or {}
    n_pages = len(pages) if isinstance(pages, dict) else 0
    expected = max(1, n_pages * 1500)
    density = min(1.0, len(plain_text) / expected)
    if density < 0.3:
        reasons.append(f"text density {density:.2f} (sparse -- likely scanned)")

    score = min(letter_ratio * 1.1, 1.0) * 0.5 + (1.0 - short_ratio) * 0.3 + density * 0.2
    return _clip(score), reasons


def build_envelope_dict(record_flat: dict[str, Any]) -> dict[str, Any]:
    """Transform the flat ``process_pdf`` record into contract-``Envelope`` shape.

    Pure triage function per F19 section 2 tables: drops pipeline-internal
    keys, nests issue fields, renames ``gazette_notices`` -> ``notices``,
    synthesizes sentinel ``scope``/``provenance`` for every corrigendum (each
    paired with a ``corrigendum_scope_defaulted`` warning bridging to F31),
    coerces any ``type="table"`` body segments to text (tracked via a
    ``table_coerced_to_text`` warning), and stamps
    ``output_format_version=1``. Raises ``KeyError`` on any unknown top-level
    key so contract drift cannot leak silently. No I/O.
    """
    _DROP = {
        "pdf_title", "pdf_file_name", "pdf_path", "pdf_size_bytes",
        "pages", "docling", "gazette_issue_id",
    }
    _PASS = {
        "pdf_sha256", "library_version", "schema_version", "extracted_at",
        "warnings", "document_confidence", "layout_info",
    }
    _ISSUE = {
        "volume", "issue_no", "publication_date", "supplement_no",
        "masthead_text", "parse_confidence",
    }
    _OTHER = {"corrigenda", "gazette_notices"}

    expected = _DROP | _PASS | _ISSUE | _OTHER
    unknown = set(record_flat.keys()) - expected
    if unknown:
        raise KeyError(
            f"build_envelope_dict: unknown top-level key(s) {sorted(unknown)}"
        )

    warnings_out: list[dict[str, Any]] = [
        dict(w) for w in (record_flat.get("warnings") or [])
    ]

    corrigenda_out: list[dict[str, Any]] = []
    for cor in record_flat.get("corrigenda") or []:
        target_notice_no = cor.get("referenced_notice_no")
        target_year_raw = cor.get("referenced_year")
        target_year: int | None
        if target_year_raw is None:
            target_year = None
        else:
            target_year = int(target_year_raw)
        corrigenda_out.append({
            "scope": "notice_references_other",
            "raw_text": cor.get("raw_text", ""),
            "provenance": {
                "header_match": "inferred",
                "line_span": [0, 0],
                "raw_header_line": None,
                "stitched_from": [],
            },
            "target_notice_no": target_notice_no,
            "target_year": target_year,
            "amendment": cor.get("correction_text"),
        })
        warnings_out.append({
            "kind": "corrigendum_scope_defaulted",
            "message": (
                "Corrigendum scope and provenance defaulted; real extraction "
                "deferred to F31"
            ),
            "where": {"notice_no": target_notice_no, "page_no": None},
        })

    notices_out: list[dict[str, Any]] = []
    for n in record_flat.get("gazette_notices") or []:
        n_out = dict(n)
        segments_out: list[dict[str, Any]] = []
        for seg in n_out.get("body_segments") or []:
            if seg.get("type") == "table":
                segments_out.append({
                    "type": "text",
                    "lines": list(seg.get("raw_lines") or []),
                })
                warnings_out.append({
                    "kind": "table_coerced_to_text",
                    "message": (
                        "Table body segment coerced to text; richer table "
                        "segment type deferred to roadmap M5"
                    ),
                    "where": {
                        "notice_no": n_out.get("gazette_notice_no"),
                        "notice_id": n_out.get("notice_id"),
                    },
                })
            else:
                segments_out.append(seg)
        n_out["body_segments"] = segments_out
        notices_out.append(n_out)

    issue = {
        "gazette_issue_id": record_flat["gazette_issue_id"],
        "masthead_text": record_flat["masthead_text"],
        "parse_confidence": record_flat["parse_confidence"],
        "volume": record_flat.get("volume"),
        "issue_no": record_flat.get("issue_no"),
        "publication_date": record_flat.get("publication_date"),
        "supplement_no": record_flat.get("supplement_no"),
    }

    # The notebook's layout computation emits a top-level ``n_pages`` tally
    # that is redundant with ``len(pages)`` and not part of ``LayoutInfo``;
    # prune to the two contract fields. (Documented as a discrepancy in the
    # F19 build report.)
    layout_info_src = record_flat["layout_info"]
    layout_info = {
        "layout_confidence": layout_info_src["layout_confidence"],
        "pages": list(layout_info_src.get("pages") or []),
    }

    return {
        "library_version": record_flat["library_version"],
        "schema_version": record_flat["schema_version"],
        "output_format_version": 1,
        "extracted_at": record_flat["extracted_at"],
        "pdf_sha256": record_flat["pdf_sha256"],
        "issue": issue,
        "notices": notices_out,
        "corrigenda": corrigenda_out,
        "document_confidence": record_flat["document_confidence"],
        "layout_info": layout_info,
        "warnings": warnings_out,
    }


@dataclass
class GazettePipeline:
    converter: DocumentConverter = field(default_factory=DocumentConverter)
    include_full_docling_dict: bool = False

    def process_pdf(self, pdf_path: Path) -> dict[str, Any]:
        # F14: Capture extraction timestamp at the very top
        extracted_at = make_extracted_at()
        
        pdf_path = Path(pdf_path).resolve()
        stat = pdf_path.stat()

        result = self.converter.convert(str(pdf_path))
        doc = result.document

        title_guess = extract_title_from_docling(doc) or pdf_path.stem.replace("_", " ")
        page_count = len(doc.pages) if getattr(doc, "pages", None) else None

        plain = doc.export_to_text()
        md = doc.export_to_markdown()

        doc_dict = doc.export_to_dict()

        # Spatial reorder: reconstruct plain text using bbox coordinates
        # so two-column pages read left-col then right-col then full-width.
        # Also returns a band-level layout confidence report.
        plain_spatial, layout_info = reorder_by_spatial_position_with_confidence(doc_dict)

        # F11: Parse masthead for identity fields
        masthead_data = parse_masthead(plain_spatial)

        ocr_score, ocr_reasons = _estimate_ocr_quality(doc_dict, plain_spatial)

        notices = split_gazette_notices(plain_spatial)
        # If the document is a scanned pre-2010 gazette, cap boundary
        # confidence before scoring so the composite reflects that. We do
        # this by marking header_match as "inferred" when it was strict but
        # OCR quality is very poor.
        if ocr_score < 0.5 and notices:
            for n in notices:
                prov = n.get("provenance") or {}
                if prov.get("header_match") == "strict":
                    prov["ocr_quality"] = ocr_score
                n["provenance"] = prov
        notices = score_notices(notices)
        if ocr_score < 0.5:
            for n in notices:
                scores = n.get("confidence_scores") or {}
                if "boundary" in scores and scores["boundary"] > 0.6:
                    scores["boundary"] = 0.6
                    scores["composite"] = round(composite_confidence(scores), 3)
                    n.setdefault("confidence_reasons", []).append(
                        "boundary: capped at 0.6 due to low OCR quality"
                    )
                    n["confidence_scores"] = scores

        # F13: Compute pdf_sha256 and gazette_issue_id (after scoring, before stamping)
        pdf_sha256 = compute_pdf_sha256(pdf_path)
        gazette_issue_id, is_fallback = make_gazette_issue_id(masthead_data, pdf_sha256)
        
        # Initialize warnings list
        warnings: list[dict[str, Any]] = []
        if is_fallback:
            warnings.append({
                "kind": "masthead.parse_failed",
                "message": "Required masthead field missing; using fallback issue id",
                "where": {"pdf_file_name": pdf_path.name}
            })

        # F13: Stamp identity fields onto notices
        for notice in notices:
            notice["gazette_issue_id"] = gazette_issue_id
            provenance = notice.get("provenance") or {}
            line_span = provenance.get("line_span", [0, 0])
            line_span_start = line_span[0] if line_span else 0
            notice["notice_id"] = make_notice_id(
                gazette_issue_id,
                notice.get("gazette_notice_no"),
                line_span_start
            )

        # F19: Stamp content_sha256 (Notice.content_sha256 is a required
        # contract field; computed from the notice's full text so it is a
        # stable content fingerprint independent of the extraction run).
        for notice in notices:
            notice["content_sha256"] = hashlib.sha256(
                notice["gazette_notice_full_text"].encode("utf-8")
            ).hexdigest()

        # F13: Stamp gazette_issue_id onto corrigenda
        corrigenda = extract_corrigenda(plain_spatial)
        if isinstance(corrigenda, list):
            for corrigendum in corrigenda:
                if isinstance(corrigendum, dict):
                    corrigendum["gazette_issue_id"] = gazette_issue_id

        document_confidence = compute_document_confidence(
            notices,
            layout_confidence=layout_info.get("layout_confidence", 1.0),
            ocr_quality=ocr_score,
        )
        document_confidence["ocr_reasons"] = ocr_reasons

        # Calculate parse confidence for masthead (1.0=all good, 0.5=partial, 0.0=all None)
        parsed_fields = sum(1 for v in masthead_data.values() if v is not None)
        parse_confidence = 1.0 if parsed_fields >= 3 else (0.5 if parsed_fields >= 1 else 0.0)

        record: dict[str, Any] = {
            "pdf_title": title_guess,
            "pdf_file_name": pdf_path.name,
            "pdf_path": str(pdf_path),
            "pdf_size_bytes": stat.st_size,
            "pdf_sha256": pdf_sha256,
            "gazette_issue_id": gazette_issue_id,
            "library_version": LIBRARY_VERSION,
            "schema_version": SCHEMA_VERSION,
            "extracted_at": extracted_at,
            "warnings": warnings,
            "pages": page_count,
            "volume": masthead_data.get("volume"),
            "issue_no": masthead_data.get("issue_no"),
            "publication_date": masthead_data.get("publication_date"),
            "supplement_no": masthead_data.get("supplement_no"),
            "masthead_text": "\n".join(plain_spatial.split("\n")[:30]),
            "parse_confidence": parse_confidence,
            "document_confidence": document_confidence,
            "layout_info": layout_info,
            "docling": {
                "export_summary": docling_export_summary(doc_dict),
                "full_markdown": md,
                "full_plain_text": plain,
                "full_plain_text_spatial": plain_spatial,
            },
            "corrigenda": corrigenda,
            "gazette_notices": notices,
        }

        if self.include_full_docling_dict:
            record["docling"]["full_docling_document_dict"] = doc_dict

        # F19: flatten -> nested-shape adapter -> strict Pydantic validation.
        # ValidationError is intentionally NOT caught: no JSON file is
        # written on a failed validation, and the caller sees the error
        # verbatim. The validated model dump becomes the on-disk payload
        # so future loads round-trip cleanly through Envelope.model_validate.
        record_nested = build_envelope_dict(record)
        env = Envelope.model_validate(record_nested)
        record = env.model_dump(mode="json")

        # Create subdirectory for this PDF's outputs
        pdf_output_dir = OUTPUT_DIR / pdf_path.stem
        pdf_output_dir.mkdir(parents=True, exist_ok=True)

        out_json = pdf_output_dir / f"{pdf_path.stem}_gazette_spatial.json"
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        out_spatial_txt = pdf_output_dir / f"{pdf_path.stem}_spatial.txt"
        with open(out_spatial_txt, "w", encoding="utf-8") as f:
            f.write(plain_spatial)

        out_markdown_default = pdf_output_dir / f"{pdf_path.stem}_docling_markdown.md"
        with open(out_markdown_default, "w", encoding="utf-8") as f:
            f.write(highlight_gazette_notices_in_markdown(md))

        out_markdown_spatial = pdf_output_dir / f"{pdf_path.stem}_spatial_markdown.md"
        with open(out_markdown_spatial, "w", encoding="utf-8") as f:
            f.write(highlight_gazette_notices_in_markdown(plain_spatial))

        docling_only = pdf_output_dir / f"{pdf_path.stem}_docling.json"
        with open(docling_only, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

        print(f"Wrote: {out_json}")
        print(f"Wrote: {out_spatial_txt}")
        print(f"Wrote: {out_markdown_default}")
        print(f"Wrote: {out_markdown_spatial}")
        print(f"Wrote: {docling_only}")
        return record


def run_pdfs(pipeline: GazettePipeline, pdf_paths: list[Path]) -> list[dict[str, Any]]:
    """Run the pipeline on an explicit ordered list of PDF paths."""
    if not pdf_paths:
        print("No PDF paths to process.")
        return []
    results: list[dict[str, Any]] = []
    for p in pdf_paths:
        print("\n--- Processing:", p.name, "---")
        results.append(pipeline.process_pdf(p))
    return results


def run_folder(pipeline: GazettePipeline, folder: Path | None = None) -> list[dict[str, Any]]:
    """Process every *.pdf in folder (same as selection mode \"all\")."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    return run_pdfs(pipeline, pdfs)


def resolve_pdf_selection(
    mode: str,
    selected_names: list[str],
    folder: Path | None = None,
) -> list[Path]:
    """Resolve \"all\" or \"selected\" to a list of existing PDF paths under folder."""
    folder = Path(folder or PDF_DIR)
    pdfs = sorted(folder.glob("*.pdf"))
    if not pdfs:
        print(f"No PDF files in {folder}. Add .pdf files and re-run.")
        return []
    m = (mode or "all").strip().lower()
    if m == "all":
        return pdfs
    if m == "selected":
        if not selected_names:
            print("PDF_SELECTION_MODE is 'selected' but SELECTED_PDF_NAMES is empty. Nothing to do.")
            return []
        by_name = {p.name: p for p in pdfs}
        out: list[Path] = []
        missing: list[str] = []
        for raw in selected_names:
            name = raw.strip()
            if not name:
                continue
            p = by_name.get(name)
            if p is None:
                p = by_name.get(Path(name).name)
            if p is None:
                missing.append(name)
            elif p not in out:
                out.append(p)
        if missing:
            print("Warning: not found in", folder, ":", missing)
            print("Available PDFs:", [p.name for p in pdfs])
        return out
    raise ValueError('PDF_SELECTION_MODE must be "all" or "selected".')

## Choose which PDFs to process

Set **`PDF_SELECTION_MODE`** in the next cell:

- **`"all"`** — every `*.pdf` under `PDF_DIR`
- **`"selected"`** — only files listed in **`SELECTED_PDF_NAMES`** (exact file names as they appear in `pdfs/`)

Then run the pipeline cell.

## Run

First conversion may **download models** and take longer. Subsequent runs are faster.

In [10]:
# --- Configure which PDFs to process ---
# When mode is "selected", list exact file names as they appear in PDF_DIR (order is preserved).
# When mode is "all", SELECTED_PDF_NAMES is ignored.
PDF_SELECTION_MODE = "selected"  # "all" | "selected"
SELECTED_PDF_NAMES: list[str] = [
    # "Kenya Gazette Vol CXINo 108.pdf",
    # "Kenya Gazette Vol CXINo 107.pdf",
    # "Kenya Gazette Vol CXINo 106.pdf",
    # "Kenya Gazette Vol CXINo 105.pdf",
    # "Kenya Gazette Vol CXINo 104.pdf",
    # "Kenya Gazette Vol CXINo 103.pdf",
    # "Kenya Gazette Vol CXINo 102.pdf",
    # "Kenya Gazette Vol CXINo 101.pdf",
    "Kenya Gazette Vol CXINo 100.pdf",
    # "Kenya Gazette Vol CXIINo 136.pdf",
    # "Kenya Gazette Vol CXIINo 132.pdf",
    # "Kenya Gazette Vol CXIINo 76.pdf",
    # "Kenya Gazette Vol CVIINo 90.pdf",
    "Kenya Gazette Vol CIINo 83 - pre 2010.pdf",
    # "Kenya Gazette Vol CXXVIIINo 41.pdf",
    # "Kenya Gazette Vol CXXVIIINo 51.pdf",
    # "Kenya Gazette Vol CXXVIINo 255.pdf",
    # "Kenya Gazette Vol CXXVIINo 254.pdf",
    # "Kenya Gazette Vol CXXVIINo 258.pdf",
    # "Kenya Gazette Vol CXXVIINo 107.pdf",
    # "Kenya Gazette Vol CXXVIINo 108.pdf",
    # "Kenya Gazette Vol CXXVIINo 111.pdf",
    "Kenya Gazette Vol CXXVIINo 63.pdf",
    # "Kenya Gazette Vol CXXVIINo 72.pdf",
    # "Kenya Gazette Vol CXXVIINo 73.pdf",
    # "Kenya Gazette Vol CXXVIINo 77.pdf",
    # "Kenya Gazette Vol CXXVIINo 266.pdf",
    "Kenya Gazette Vol CXXIVNo 282.pdf"
]

pipeline = GazettePipeline(include_full_docling_dict=False)
pdf_paths = resolve_pdf_selection(PDF_SELECTION_MODE, SELECTED_PDF_NAMES)
print(
    "Mode:",
    PDF_SELECTION_MODE,
    "|",
    len(pdf_paths),
    "file(s):",
    [p.name for p in pdf_paths],
)
results = run_pdfs(pipeline, pdf_paths)

if results:
    sample = results[0]
    notices = sample.get("gazette_notices") or []
    corrigenda = sample.get("corrigenda") or []
    with_tables = [n for n in notices if "derived_table" in n]
    print("Keys:", list(sample.keys()))
    print("Pages:", sample.get("pages"))
    print(f"Corrigenda count: {len(corrigenda)}")
    print(f"Notice count: {len(notices)}")
    print(f"Notices with derived tables: {len(with_tables)}")
    if corrigenda:
        first_corr = corrigenda[0]
        print(f"\nFirst corrigendum preview:")
        print(f"  Referenced: Notice {first_corr.get('referenced_notice_no')} of {first_corr.get('referenced_year')}")
        print(f"  Error: {first_corr.get('error_text')}")
        print(f"  Correction: {first_corr.get('correction_text')}")
    if notices:
        first = notices[0]
        print(f"\nFirst notice preview:")
        print(f"  No: {first.get('gazette_notice_no')}")
        print(f"  Header: {first.get('gazette_notice_header')}")
        print(f"  Title lines: {first.get('title_lines', [])}")
        segs = first.get("body_segments", [])
        seg_types = [s["type"] for s in segs]
        print(f"  Body segments: {len(segs)} ({', '.join(f'{t}={seg_types.count(t)}' for t in dict.fromkeys(seg_types))})")
    display_snippet = json.dumps(sample, ensure_ascii=False, indent=2)
    if len(display_snippet) > 8000:
        print(display_snippet[:8000] + "\n... [truncated for display; see output JSON files] ...")
    else:
        print(display_snippet)

[INFO] 2026-04-19 22:49:18,488 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-19 22:49:18,489 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-19 22:49:18,519 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-19 22:49:18,519 [RapidOCR] main.py:50: Using C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth


Mode: selected | 1 file(s): ['Kenya Gazette Vol CXXVIINo 63.pdf']

--- Processing: Kenya Gazette Vol CXXVIINo 63.pdf ---


[INFO] 2026-04-19 22:49:18,692 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-19 22:49:18,693 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-19 22:49:18,696 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-19 22:49:18,696 [RapidOCR] main.py:50: Using C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-04-19 22:49:18,770 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-19 22:49:18,770 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-19 22:49:18,829 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-04-19 22:49:18,829 [RapidOCR] main.py:50: Using C:\

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

RapidOCR returned empty result!
Stage preprocess failed for run 1, pages [68]: std::bad_alloc
Stage preprocess failed for run 1, pages [69]: std::bad_alloc
Stage preprocess failed for run 1, pages [70]: std::bad_alloc
Stage preprocess failed for run 1, pages [71]: std::bad_alloc
Stage preprocess failed for run 1, pages [72]: std::bad_alloc
Stage preprocess failed for run 1, pages [73]: std::bad_alloc
Stage preprocess failed for run 1, pages [74]: std::bad_alloc
Stage preprocess failed for run 1, pages [75]: std::bad_alloc
Stage preprocess failed for run 1, pages [76]: std::bad_alloc
Stage preprocess failed for run 1, pages [77]: std::bad_alloc
Stage preprocess failed for run 1, pages [78]: std::bad_alloc
Stage preprocess failed for run 1, pages [79]: std::bad_alloc
Stage preprocess failed for run 1, pages [80]: std::bad_alloc
Stage preprocess failed for run 1, pages [81]: std::bad_alloc
Stage preprocess failed for run 1, pages [82]: std::bad_alloc
Stage preprocess failed for run 1, pag

Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_gazette_spatial.json
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_spatial.txt
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_docling_markdown.md
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_spatial_markdown.md
Wrote: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\output\Kenya Gazette Vol CXXVIINo 63\Kenya Gazette Vol CXXVIINo 63_docling.json
Keys: ['pdf_title', 'pdf_file_name', 'pdf_path', 'pdf_size_bytes', 'pdf_sha256', 'gazette_issue_id', 'library_version', 'schema_version', 'extracted_at', 'warnings', 'pages', 'volume', 'issue_no', 'publication_date', 'supplement_no', 'masthead_text', 'parse_confidence', 'do

## Optional: embed full Docling tree in the main record

Set `include_full_docling_dict=True` if you need the complete `DoclingDocument` as JSON inside `gazette.json` (large).

In [ ]:
# Example: toggle full dict for a single file
# p = PDF_DIR / "your_issue.pdf"
# if p.exists():
#     full_pipeline = GazettePipeline(include_full_docling_dict=True)
#     full_pipeline.process_pdf(p)

## Confidence report

Walks the `output/` tree and produces a triage view:

- Per-PDF table: notice count, counts by band (high / medium / low), mean and min composite.
- Top-N lowest-confidence notices across the corpus with their reasons.
- Optional CSV export to `output/_confidence_report.csv`.

Run after processing one or more PDFs. If a JSON predates the scoring integration it simply shows as unscored.

In [ ]:
import csv


def _iter_output_gazette_jsons(output_root: Path = OUTPUT_DIR):
    """Yield (pdf_stem, json_path, record) for every gazette_spatial.json in output/."""
    for path in sorted(output_root.glob("*/*_gazette_spatial.json")):
        try:
            with open(path, "r", encoding="utf-8") as f:
                record = json.load(f)
        except (json.JSONDecodeError, OSError) as exc:
            print(f"skip {path.name}: {exc}")
            continue
        yield path.parent.name, path, record


def confidence_report(
    output_root: Path = OUTPUT_DIR,
    top_n_low: int = 20,
    csv_path: Path | None = None,
) -> dict[str, Any]:
    """Summarize confidence across every processed gazette JSON under output_root.

    Prints a per-PDF table and the lowest-confidence notices across the corpus.
    Returns a dict with the numeric summary for programmatic use.
    """
    per_pdf: list[dict[str, Any]] = []
    all_notices: list[tuple[str, dict[str, Any]]] = []

    for stem, path, record in _iter_output_gazette_jsons(output_root):
        notices = record.get("gazette_notices") or []
        scored = [n for n in notices if (n.get("confidence_scores") or {}).get("composite") is not None]
        bands = partition_by_band(notices)
        agg = aggregate_confidence(notices)
        doc_conf = record.get("document_confidence") or {}
        per_pdf.append({
            "pdf": stem,
            "n_notices": len(notices),
            "scored": len(scored),
            "high": len(bands["high"]),
            "medium": len(bands["medium"]),
            "low": len(bands["low"]),
            "mean_composite": agg["mean"],
            "min_composite": agg["min"],
            "layout": doc_conf.get("layout"),
            "ocr_quality": doc_conf.get("ocr_quality"),
            "document_composite": doc_conf.get("composite"),
        })
        for n in scored:
            all_notices.append((stem, n))

    print(
        f"{'PDF':<55} {'N':>4} {'High':>5} {'Med':>4} {'Low':>4} "
        f"{'Mean':>6} {'Min':>6} {'Layout':>7} {'OCR':>5}"
    )
    print("-" * 100)
    for row in per_pdf:
        print(
            f"{row['pdf'][:55]:<55} {row['n_notices']:>4} "
            f"{row['high']:>5} {row['medium']:>4} {row['low']:>4} "
            f"{row['mean_composite']:>6.3f} {row['min_composite']:>6.3f} "
            f"{(row['layout'] or 0):>7.3f} {(row['ocr_quality'] or 0):>5.3f}"
        )

    all_notices.sort(key=lambda t: (t[1].get("confidence_scores") or {}).get("composite", 1.0))
    print(f"\nLowest {min(top_n_low, len(all_notices))} notices across corpus:")
    print("-" * 100)
    for stem, n in all_notices[:top_n_low]:
        c = (n.get("confidence_scores") or {}).get("composite")
        no = n.get("gazette_notice_no") or "?"
        hdr = (n.get("gazette_notice_header") or "")[:60]
        reasons = n.get("confidence_reasons") or []
        print(f"[{c:.3f}] {stem[:40]:<40} #{no:<8} {hdr}")
        for r in reasons[:3]:
            print(f"         {r}")

    if csv_path is not None:
        csv_path = Path(csv_path)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow([
                "pdf", "notice_no", "composite", "notice_number", "structure",
                "spatial", "boundary", "table", "header_match", "reasons",
            ])
            for stem, n in all_notices:
                s = n.get("confidence_scores") or {}
                prov = n.get("provenance") or {}
                w.writerow([
                    stem,
                    n.get("gazette_notice_no"),
                    s.get("composite"),
                    s.get("notice_number"),
                    s.get("structure"),
                    s.get("spatial"),
                    s.get("boundary"),
                    s.get("table"),
                    prov.get("header_match"),
                    " | ".join(n.get("confidence_reasons") or []),
                ])
        print(f"\nWrote CSV: {csv_path}")

    return {"per_pdf": per_pdf, "n_notices_total": len(all_notices)}


# Uncomment to run a report over every processed PDF:
_rep = confidence_report(csv_path=OUTPUT_DIR / "_confidence_report.csv")

## Optional: LLM semantic validation

For notices whose rule-based composite is below a threshold (default 0.70) a lightweight LLM can check for scrambled text, merged notices, truncation, and legal incoherence -- things regex cannot see.

- Set `ENABLE_LLM_VALIDATION = True` below.
- Requires `OPENAI_API_KEY` (for OpenAI) or `ANTHROPIC_API_KEY` (for Claude) in the environment.
- Results are cached by body hash under `.llm_cache/` so reruns are free.
- Writes `llm_validation` on each scored notice, plus `composite_enhanced` and `needs_human_review`.

Call `enhance_with_llm(notices)` on any already-scored notice list to update it in place.

In [ ]:
# %pip install openai

In [11]:
import hashlib
import os

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass

ENABLE_LLM_VALIDATION = True
LLM_CONFIDENCE_THRESHOLD = 0.80
LLM_MODEL = "gpt-4o-mini"
LLM_CACHE_DIR = PROJECT_ROOT / ".llm_cache"

_LLM_PROMPT = """You are validating an extracted Kenya Gazette notice.
Check for these issues:
1. Text coherence -- does the text flow naturally or appear scrambled from column-mixing?
2. Completeness -- does the notice end with a date and signature, or is it truncated?
3. Single notice -- is this one notice, or were multiple notices merged together?
4. Legal structure -- does it follow gazette patterns (preamble, body, closing)?

Notice:
---
{body}
---

Respond with JSON only:
{{
  "coherence_score": <0.0-1.0>,
  "completeness_score": <0.0-1.0>,
  "single_notice_score": <0.0-1.0>,
  "legal_structure_score": <0.0-1.0>,
  "issues": [<short strings>],
  "needs_human_review": <true/false>
}}
"""


def _llm_cache_path(body: str, model: str) -> Path:
    LLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha1(f"{model}|{body[:4000]}".encode("utf-8")).hexdigest()
    return LLM_CACHE_DIR / f"{key}.json"


def _llm_call_openai(body: str, model: str) -> dict[str, Any]:
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise RuntimeError("pip install openai") from exc
    client = OpenAI()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": _LLM_PROMPT.format(body=body[:4000])}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=300,
    )
    return json.loads(resp.choices[0].message.content or "{}")


def llm_validate_notice(notice: dict[str, Any], model: str = LLM_MODEL) -> dict[str, Any]:
    """Run the LLM validator on a single notice, using the on-disk cache."""
    body = notice.get("gazette_notice_full_text") or ""
    if not body.strip():
        return {"error": "empty body", "needs_human_review": True}

    cache_path = _llm_cache_path(body, model)
    if cache_path.exists():
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except (json.JSONDecodeError, OSError):
            pass

    try:
        result = _llm_call_openai(body, model)
    except Exception as exc:
        # Do not cache transport / config errors -- they are not real LLM
        # judgments and would poison subsequent runs until cleared.
        return {"error": str(exc), "needs_human_review": True}

    if "error" not in result:
        try:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
        except OSError:
            pass
    return result


def enhance_with_llm(
    notices: list[dict[str, Any]],
    threshold: float = LLM_CONFIDENCE_THRESHOLD,
    model: str = LLM_MODEL,
) -> list[dict[str, Any]]:
    """For every notice with composite < threshold, run LLM validation and
    blend the result into composite_enhanced. Marks needs_human_review when
    any LLM sub-score is below 0.5.
    """
    for n in notices:
        scores = n.get("confidence_scores") or {}
        composite = scores.get("composite")
        if composite is None or composite >= threshold:
            continue
        res = llm_validate_notice(n, model=model)
        n["llm_validation"] = res
        sub = [
            res.get("coherence_score"),
            res.get("completeness_score"),
            res.get("single_notice_score"),
            res.get("legal_structure_score"),
        ]
        sub_num = [float(x) for x in sub if isinstance(x, (int, float))]
        if sub_num:
            llm_avg = sum(sub_num) / len(sub_num)
            scores["llm_semantic"] = round(llm_avg, 3)
            scores["composite_enhanced"] = round(
                composite * 0.6 + llm_avg * 0.4, 3
            )
            n["confidence_scores"] = scores
            if any(s < 0.5 for s in sub_num) or res.get("needs_human_review"):
                res["needs_human_review"] = True
    return notices


if ENABLE_LLM_VALIDATION and os.environ.get("OPENAI_API_KEY"):
    print("LLM validation enabled. Call enhance_with_llm(notices) after scoring.")
else:
    print("LLM validation defined but not active. Set ENABLE_LLM_VALIDATION=True "
          "and export OPENAI_API_KEY to use it.")

LLM validation enabled. Call enhance_with_llm(notices) after scoring.


### Run LLM validation on a single gazette

Load one already-parsed gazette, run `enhance_with_llm` only on its low-confidence notices (composite < 0.70), mirror `needs_human_review` to the notice top level so it is easy to query, and write the enhanced JSON back in place. Responses are cached under `.llm_cache/` so reruns cost nothing.

Change `target` to point at any folder in `output/`.

In [ ]:
target = "Kenya Gazette Vol CXXIVNo 282"
gpath = OUTPUT_DIR / target / f"{target}_gazette_spatial.json"

with open(gpath, "r", encoding="utf-8") as f:
    record = json.load(f)

notices = record["gazette_notices"]

below = [
    n for n in notices
    if (n.get("confidence_scores") or {}).get("composite", 1.0) < LLM_CONFIDENCE_THRESHOLD
]
print(f"{len(below)} of {len(notices)} notices below {LLM_CONFIDENCE_THRESHOLD:.2f} -- those will be sent to the LLM")

enhance_with_llm(notices)

# Mirror nested flag to the top level for easy filtering / DB ingest.
for n in notices:
    if (n.get("llm_validation") or {}).get("needs_human_review"):
        n["needs_human_review"] = True

flagged = [n for n in notices if n.get("needs_human_review")]
print(f"{len(flagged)} notices flagged for human review")

with open(gpath, "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)

print(f"Wrote enhanced JSON back to {gpath.name}")

# Show the flagged notices so you can spot-check them.
for n in flagged[:10]:
    scores = n.get("confidence_scores") or {}
    issues = (n.get("llm_validation") or {}).get("issues") or []
    print(
        f"  - notice {n.get('gazette_notice_no')}: "
        f"composite={scores.get('composite')} "
        f"enhanced={scores.get('composite_enhanced')} "
        f"issues={issues[:3]}"
    )

1 of 1 notices below 0.80 -- those will be sent to the LLM
1 notices flagged for human review
Wrote enhanced JSON back to Kenya Gazette Vol CIINo 83 - pre 2010_gazette_spatial.json
  - notice None: composite=0.253 enhanced=0.282 issues=['Text appears scrambled and incoherent.', 'Notice is truncated and does not end properly.', 'Multiple notices seem to be merged together.']


## Calibration and regression

Two small tools to validate the confidence system:

1. **Calibration** -- `sample_for_calibration()` draws a stratified sample of ~80-100 notices across the four bands and writes a YAML stub at `tests/calibration_sample.yaml`. Fill the `is_correct` field by hand for each notice, then `score_calibration()` computes precision/recall per band and prints suggested weight tweaks.

2. **Regression** -- `update_regression_fixture()` writes a snapshot of mean composite per canonical PDF to `tests/expected_confidence.json`. `check_regression()` reloads it and prints a warning when mean composite has dropped beyond the tolerance (default 0.05) for any fixture PDF.

In [20]:
import random


TESTS_DIR = PROJECT_ROOT / "tests"
CALIBRATION_FILE = TESTS_DIR / "calibration_sample.yaml"
REGRESSION_FILE = TESTS_DIR / "expected_confidence.json"

CANONICAL_PDFS = [
    "Kenya Gazette Vol CXINo 100",
    "Kenya Gazette Vol CXINo 103",
    "Kenya Gazette Vol CXIINo 76",
    "Kenya Gazette Vol CXXVIINo 63",
    "Kenya Gazette Vol CXXIVNo 282",
    "Kenya Gazette Vol CIINo 83 - pre 2010",
]


def sample_for_calibration(
    n_per_band: int = 20,
    seed: int = 42,
    output_root: Path = OUTPUT_DIR,
    out_path: Path = CALIBRATION_FILE,
) -> Path:
    """Draw a stratified sample across confidence bands and emit a YAML stub
    the user can hand-label with `is_correct: true/false` per notice.
    """
    random.seed(seed)
    bands: dict[str, list[dict[str, Any]]] = {"high": [], "medium": [], "low": []}
    for stem, _path, record in _iter_output_gazette_jsons(output_root):
        for n in record.get("gazette_notices") or []:
            c = (n.get("confidence_scores") or {}).get("composite")
            if c is None:
                continue
            band = "high" if c >= 0.80 else "medium" if c >= 0.50 else "low"
            bands[band].append({"pdf": stem, "notice": n})

    out_path.parent.mkdir(parents=True, exist_ok=True)
    lines: list[str] = [
        "# Calibration sample for confidence scoring.",
        "# For each entry, set is_correct to true or false after manual review.",
        "# Then run score_calibration() to compute precision/recall per band.",
        "",
    ]
    total = 0
    for band in ("high", "medium", "low"):
        pool = bands[band]
        pick = random.sample(pool, min(n_per_band, len(pool)))
        for item in pick:
            n = item["notice"]
            scores = n.get("confidence_scores") or {}
            header = (n.get("gazette_notice_header") or "").replace('"', "'")[:80]
            lines.append(f"- pdf: \"{item['pdf']}\"")
            lines.append(f"  notice_no: \"{n.get('gazette_notice_no')}\"")
            lines.append(f"  header: \"{header}\"")
            lines.append(f"  composite: {scores.get('composite')}")
            lines.append(f"  band: {band}")
            lines.append(f"  is_correct: null  # true | false")
            lines.append("")
            total += 1

    out_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"Wrote {total} notices to {out_path}")
    print("Edit the file to set is_correct for each, then call score_calibration().")
    return out_path


def _parse_calibration_yaml(path: Path) -> list[dict[str, Any]]:
    """Minimal YAML-subset parser -- we only emit the shape sample_for_calibration
    writes, so a dependency on PyYAML is unnecessary.
    """
    text = path.read_text(encoding="utf-8")
    entries: list[dict[str, Any]] = []
    current: dict[str, Any] | None = None
    for raw_line in text.splitlines():
        line = raw_line.rstrip()
        if not line or line.lstrip().startswith("#"):
            continue
        if line.startswith("- "):
            if current:
                entries.append(current)
            current = {}
            rest = line[2:]
            if ":" in rest:
                k, _, v = rest.partition(":")
                current[k.strip()] = _coerce(v.strip())
        elif current is not None and ":" in line:
            k, _, v = line.partition(":")
            current[k.strip()] = _coerce(v.strip())
    if current:
        entries.append(current)
    return entries


def _coerce(v: str) -> Any:
    if v.startswith('"') and v.endswith('"'):
        return v[1:-1]
    if v == "null":
        return None
    if v == "true":
        return True
    if v == "false":
        return False
    try:
        return float(v) if "." in v else int(v)
    except ValueError:
        return v


def score_calibration(path: Path = CALIBRATION_FILE) -> dict[str, Any]:
    """Compute precision/recall per band from a hand-labeled calibration file."""
    if not path.exists():
        print(f"{path} not found. Run sample_for_calibration() first.")
        return {}
    entries = _parse_calibration_yaml(path)
    labeled = [e for e in entries if isinstance(e.get("is_correct"), bool)]
    if not labeled:
        print("No entries have is_correct set. Fill in the file first.")
        return {}

    summary: dict[str, dict[str, int]] = {}
    for e in labeled:
        band = e.get("band", "?")
        summary.setdefault(band, {"n": 0, "correct": 0, "wrong": 0})
        summary[band]["n"] += 1
        if e["is_correct"]:
            summary[band]["correct"] += 1
        else:
            summary[band]["wrong"] += 1

    print(f"Labeled {len(labeled)}/{len(entries)} entries.")
    print(f"{'band':<8} {'n':>4} {'correct':>8} {'wrong':>6} {'precision':>10}")
    for band in ("high", "medium", "low"):
        s = summary.get(band)
        if not s:
            continue
        prec = s["correct"] / s["n"] if s["n"] else 0.0
        print(f"{band:<8} {s['n']:>4} {s['correct']:>8} {s['wrong']:>6} {prec:>10.2%}")

    hi = summary.get("high", {"n": 0, "correct": 0})
    if hi["n"] and hi["correct"] / hi["n"] < 0.85:
        print("\nHigh band precision below 85% -- consider tightening scorers:")
        print("  - raise structure deductions, or")
        print("  - lower the composite weight on notice_number.")
    low = summary.get("low", {"n": 0, "correct": 0})
    if low["n"] and low["correct"] / low["n"] > 0.3:
        print("\nLow band has many correct notices -- scorers are too strict.")
        print("  Consider relaxing structure or spatial deductions.")

    return summary


def update_regression_fixture(
    canonical: list[str] = CANONICAL_PDFS,
    out_path: Path = REGRESSION_FILE,
    output_root: Path = OUTPUT_DIR,
) -> Path:
    """Snapshot mean composite per canonical PDF for regression checks."""
    snapshot: dict[str, Any] = {}
    for name in canonical:
        path = output_root / name / f"{name}_gazette_spatial.json"
        if not path.exists():
            snapshot[name] = {"present": False}
            continue
        with open(path, "r", encoding="utf-8") as f:
            rec = json.load(f)
        agg = aggregate_confidence(rec.get("notices") or rec.get("gazette_notices") or [])
        doc_conf = rec.get("document_confidence") or {}
        snapshot[name] = {
            "present": True,
            "mean_composite": agg["mean"],
            "min_composite": agg["min"],
            "layout": doc_conf.get("layout"),
            "ocr_quality": doc_conf.get("ocr_quality"),
            "n_notices": agg["n"],
        }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    print(f"Wrote regression fixture: {out_path}")
    return out_path


def check_regression(
    tolerance: float = 0.05,
    fixture_path: Path = REGRESSION_FILE,
    output_root: Path = OUTPUT_DIR,
) -> bool:
    """Compare current mean composite per canonical PDF against the fixture.
    Returns True if all within tolerance; prints warnings otherwise.
    """
    if not fixture_path.exists():
        print(f"No fixture at {fixture_path}. Call update_regression_fixture() first.")
        return True
    expected = json.loads(fixture_path.read_text(encoding="utf-8"))
    regressed = False
    for name, snap in expected.items():
        if not snap.get("present"):
            continue
        cur_path = output_root / name / f"{name}_gazette_spatial.json"
        if not cur_path.exists():
            print(f"  missing current output for {name}")
            continue
        with open(cur_path, "r", encoding="utf-8") as f:
            rec = json.load(f)
        cur = aggregate_confidence(rec.get("notices") or rec.get("gazette_notices") or [])
        delta = cur["mean"] - float(snap["mean_composite"])
        if delta < -tolerance:
            regressed = True
            print(
                f"REGRESSION: {name} mean composite {snap['mean_composite']:.3f} -> {cur['mean']:.3f} "
                f"(drop {-delta:.3f})"
            )
        else:
            print(f"OK: {name} mean composite {cur['mean']:.3f} (baseline {snap['mean_composite']:.3f})")
    return not regressed


# Uncomment to generate a calibration sample or regression fixture.
# `sample_for_calibration()` overwrites tests/calibration_sample.yaml from
# scratch -- keep it commented so re-running the notebook does not erase
# any is_correct labels you have added by hand.
# sample_for_calibration()
update_regression_fixture()
# check_regression()

Wrote regression fixture: C:\Users\Ronald Wahome\Documents\docling_spatial_pdfs\tests\expected_confidence.json


WindowsPath('C:/Users/Ronald Wahome/Documents/docling_spatial_pdfs/tests/expected_confidence.json')

In [21]:
check_regression()

OK: Kenya Gazette Vol CXINo 100 mean composite 0.990 (baseline 0.990)
OK: Kenya Gazette Vol CXINo 103 mean composite 0.989 (baseline 0.989)
OK: Kenya Gazette Vol CXIINo 76 mean composite 0.963 (baseline 0.963)
OK: Kenya Gazette Vol CXXVIINo 63 mean composite 0.977 (baseline 0.977)
OK: Kenya Gazette Vol CXXIVNo 282 mean composite 0.968 (baseline 0.968)
OK: Kenya Gazette Vol CIINo 83 - pre 2010 mean composite 0.253 (baseline 0.253)


True

In [ ]:
# F12 Test: Trailing Content Detector

print("=" * 70)
print("F12 TRAILING CONTENT DETECTOR - TEST SUITE")
print("=" * 70)

# T1: Happy path - pricing table at end
print("\n[T1] Testing Kenya Gazette Vol CXXIVNo 282 (trailing content expected)")
t1_path = OUTPUT_DIR / "Kenya Gazette Vol CXXIVNo 282" / "Kenya Gazette Vol CXXIVNo 282_spatial.txt"
if t1_path.exists():
    t1_text = t1_path.read_text(encoding="utf-8")
    t1_notices = split_gazette_notices(t1_text)
    
    last_notice = t1_notices[-1] if t1_notices else None
    if last_notice:
        full_text = last_notice.get("gazette_notice_full_text", "")
        has_subscription = "SUBSCRIPTION AND ADVERTISEMENT CHARGES" in full_text.upper()
        has_price = "PRICE: KSH" in full_text.upper()
        
        print(f"  Total notices: {len(t1_notices)}")
        print(f"  Last notice number: {last_notice.get('gazette_notice_no')}")
        print(f"  Last notice length: {len(full_text)} chars")
        print(f"  Contains 'SUBSCRIPTION AND ADVERTISEMENT CHARGES': {has_subscription}")
        print(f"  Contains 'PRICE: KSH': {has_price}")
        
        if not has_subscription and not has_price:
            print("  ✅ T1 PASS: Trailing content successfully excluded")
        else:
            print("  ❌ T1 FAIL: Trailing content still present in last notice")
    else:
        print("  ❌ T1 FAIL: No notices found")
else:
    print(f"  ⚠️  T1 SKIP: Test file not found at {t1_path}")

# T2: Edge case - no trailing content
print("\n[T2] Testing Kenya Gazette Vol CXINo 100 (no trailing content expected)")
t2_path = OUTPUT_DIR / "Kenya Gazette Vol CXINo 100" / "Kenya Gazette Vol CXINo 100_spatial.txt"
if t2_path.exists():
    t2_text = t2_path.read_text(encoding="utf-8")
    t2_notices = split_gazette_notices(t2_text)
    
    last_notice = t2_notices[-1] if t2_notices else None
    if last_notice:
        print(f"  Total notices: {len(t2_notices)}")
        print(f"  Last notice number: {last_notice.get('gazette_notice_no')}")
        print(f"  ✅ T2 PASS: No trailing content, notices unchanged")
    else:
        print("  ❌ T2 FAIL: No notices found")
else:
    print(f"  ⚠️  T2 SKIP: Test file not found at {t2_path}")

# T3: Degraded - function should not raise
print("\n[T3] Testing degraded input (should handle gracefully)")
try:
    degraded_text = "GAZETTE NOTICE NO. 1\nSome content\n" + ("X " * 100)
    degraded_notices = split_gazette_notices(degraded_text)
    print(f"  Total notices: {len(degraded_notices)}")
    print(f"  ✅ T3 PASS: Degraded input handled gracefully (no exception)")
except Exception as e:
    print(f"  ❌ T3 FAIL: Exception raised: {e}")

# T4: Multiple trailing sections (same as T1, but verify all excluded)
print("\n[T4] Testing multiple trailing sections (all should be excluded)")
if t1_path.exists():
    patterns = [
        "SUBSCRIPTION AND ADVERTISEMENT CHARGES",
        "PRICE: KSH",
        "IMPORTANT NOTICE TO SUBSCRIBERS",
        "GOVERNMENT PRINTER"
    ]
    
    last_notice = t1_notices[-1] if t1_notices else None
    if last_notice:
        full_text = last_notice.get("gazette_notice_full_text", "").upper()
        found_patterns = [p for p in patterns if p in full_text]
        
        if not found_patterns:
            print(f"  ✅ T4 PASS: All trailing patterns excluded")
        else:
            print(f"  ❌ T4 FAIL: Found trailing patterns: {found_patterns}")
    else:
        print("  ❌ T4 FAIL: No notices found")
else:
    print(f"  ⚠️  T4 SKIP: Test file not found")

print("\n" + "=" * 70)
print("TEST SUITE COMPLETE")
print("=" * 70)

In [ ]:
# F14 Test Suite
import re
import json
from datetime import datetime, timezone
import time

print("=" * 80)
print("F14 Test Suite: Envelope Versioning Fields")
print("=" * 80)

# Test T1 & T2: Process CXXIVNo 282 and check presence + format
print("\n[T1 & T2] Processing Kenya Gazette Vol CXXIVNo 282...")
pdf_path = Path("Kenya Gazette PDFs/Kenya Gazette Vol CXXIVNo 282.pdf")
pipeline = GazettePipeline()
record1 = pipeline.process_pdf(pdf_path)

# Save output
output_dir = Path("output/Kenya Gazette Vol CXXIVNo 282")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "Kenya Gazette Vol CXXIVNo 282_gazette_spatial.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(record1, f, indent=2, ensure_ascii=False)

# T1: Check presence
required_keys = ["library_version", "schema_version", "extracted_at"]
missing = [k for k in required_keys if k not in record1]
if missing:
    print(f"  FAIL T1: Missing keys: {missing}")
else:
    print(f"  PASS T1: All three fields present")
    print(f"     library_version: {record1['library_version']}")
    print(f"     schema_version: {record1['schema_version']}")
    print(f"     extracted_at: {record1['extracted_at']}")

# T2: Format validation
lib_ver = record1["library_version"]
schema_ver = record1["schema_version"]
extracted = record1["extracted_at"]

t2_pass = True
if not re.fullmatch(r"\d+\.\d+\.\d+", lib_ver):
    print(f"  FAIL T2: library_version format invalid: {lib_ver}")
    t2_pass = False
elif not re.fullmatch(r"\d+\.\d+", schema_ver):
    print(f"  FAIL T2: schema_version format invalid: {schema_ver}")
    t2_pass = False
elif not re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z", extracted):
    print(f"  FAIL T2: extracted_at format invalid: {extracted}")
    t2_pass = False
else:
    try:
        dt = datetime.fromisoformat(extracted.replace("Z", "+00:00"))
        if dt.tzinfo != timezone.utc:
            print(f"  FAIL T2: extracted_at not UTC: {dt.tzinfo}")
            t2_pass = False
        else:
            print(f"  PASS T2: All format validations passed")
    except Exception as e:
        print(f"  FAIL T2: extracted_at not parseable: {e}")
        t2_pass = False

# Check F13 fields still present
if "pdf_sha256" in record1 and "gazette_issue_id" in record1:
    notices = record1.get("gazette_notices", [])
    if notices and all("notice_id" in n for n in notices):
        print(f"  PASS T1: F13 identity fields still intact")
    else:
        print(f"  FAIL T1: notice_id missing from some notices")
else:
    print(f"  FAIL T1: F13 fields missing")

# Test T3: Process same PDF twice - check determinism
print("\n[T3] Processing same PDF twice to test determinism...")
time.sleep(2)  # Ensure different timestamps
record2 = pipeline.process_pdf(pdf_path)

t3_pass = True
# Versions should be stable
if record1["library_version"] != record2["library_version"]:
    print(f"  FAIL T3: library_version differs: {record1['library_version']} vs {record2['library_version']}")
    t3_pass = False
if record1["schema_version"] != record2["schema_version"]:
    print(f"  FAIL T3: schema_version differs: {record1['schema_version']} vs {record2['schema_version']}")
    t3_pass = False

# extracted_at should differ
if record1["extracted_at"] == record2["extracted_at"]:
    print(f"  FAIL T3: extracted_at is identical (should differ): {record1['extracted_at']}")
    t3_pass = False

# F13 fields should match
if record1["pdf_sha256"] != record2["pdf_sha256"]:
    print(f"  FAIL T3: pdf_sha256 differs")
    t3_pass = False
if record1["gazette_issue_id"] != record2["gazette_issue_id"]:
    print(f"  FAIL T3: gazette_issue_id differs")
    t3_pass = False

notice_ids_1 = [n["notice_id"] for n in record1.get("gazette_notices", [])]
notice_ids_2 = [n["notice_id"] for n in record2.get("gazette_notices", [])]
if notice_ids_1 != notice_ids_2:
    print(f"  FAIL T3: notice_ids differ")
    t3_pass = False

if t3_pass:
    print(f"  PASS T3: versions stable, extracted_at differs, F13 ids identical")
    print(f"     Run 1 extracted_at: {record1['extracted_at']}")
    print(f"     Run 2 extracted_at: {record2['extracted_at']}")

# Test T4: Process other PDFs - cross-gazette consistency
print("\n[T4] Processing other PDFs for cross-gazette consistency...")
pdf_paths = [
    "Kenya Gazette PDFs/Kenya Gazette Vol CXINo 100.pdf",
    "Kenya Gazette PDFs/Kenya Gazette Vol CXIXNo 194.pdf",
]

records = [record1]  # Already have CXXIVNo 282
t4_pass = True

for pdf_path_str in pdf_paths:
    pdf_p = Path(pdf_path_str)
    if not pdf_p.exists():
        print(f"  WARNING: Skipping {pdf_p.name} (not found)")
        continue
    
    print(f"  Processing {pdf_p.name}...")
    r = pipeline.process_pdf(pdf_p)
    records.append(r)
    
    # Save output
    gazette_name = pdf_p.stem
    out_dir = Path(f"output/{gazette_name}")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{gazette_name}_gazette_spatial.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(r, f, indent=2, ensure_ascii=False)

# Check all have same library_version and schema_version
lib_versions = [r["library_version"] for r in records]
schema_versions = [r["schema_version"] for r in records]

if len(set(lib_versions)) != 1:
    print(f"  FAIL T4: library_version differs across PDFs: {lib_versions}")
    t4_pass = False
elif len(set(schema_versions)) != 1:
    print(f"  FAIL T4: schema_version differs across PDFs: {schema_versions}")
    t4_pass = False
elif lib_versions[0] != "0.1.0" or schema_versions[0] != "1.0":
    print(f"  FAIL T4: unexpected version values: {lib_versions[0]}, {schema_versions[0]}")
    t4_pass = False
else:
    print(f"  PASS T4: All {len(records)} gazettes share library_version='0.1.0' and schema_version='1.0'")

# Final summary
print("\n" + "=" * 80)
all_pass = t2_pass and t3_pass and t4_pass
if all_pass:
    print("ALL F14 TESTS PASSED")
else:
    print("SOME F14 TESTS FAILED")
print("=" * 80)